## From Linguistic Context to Semantic Structure

This project reads the saved arXiv dataset and computes contextual chunks, lexical weights, cluster selection, contextual profiles and supervised comparisons. Dataset acquisition is performed separately in Datasets/Datasets.ipynb. Run the cells in order; checks and diagnostic outputs remain in this notebook.


## Computational libraries

Install the libraries used by this project. This operation does not load or change shared datasets.


In [ ]:
# Purpose: install the pinned standard libraries used by this one pipeline.
import importlib.metadata
import subprocess
import sys
import shutil
REQUIRED_PACKAGES = ('nbformat==5.11.1','nbclient==0.11.0','ipykernel==7.4.0','numpy==2.3.5','scipy==1.17.0','matplotlib==3.10.8','huggingface-hub==0.36.2','scikit-learn==1.8.0','transformers==4.57.1','chonky==0.1.7','safetensors==0.8.0')
def install_notebook_dependencies(packages=REQUIRED_PACKAGES):
    missing=[]
    for package in packages:
        name,expected=package.split('==',1)
        try: installed=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: installed=None
        if installed!=expected: missing.append(package)
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],check=True)
try: torch_version=importlib.metadata.version('torch').split('+')[0]
except importlib.metadata.PackageNotFoundError: torch_version=None
if torch_version!='2.8.0':
    torch_command=[sys.executable,'-m','pip','install','--disable-pip-version-check','torch==2.8.0']
    if not shutil.which('nvidia-smi'):
        torch_command.extend(['--index-url','https://download.pytorch.org/whl/cpu'])
    subprocess.run(torch_command,check=True)
install_notebook_dependencies()


## Read the existing dataset

Read the independently prepared dataset. A missing file is caught and reported as a debug message; dependent calculations are skipped. Obtain missing files manually through Datasets/Datasets.ipynb.


In [ ]:
from pathlib import Path
import os
import json
ROOT = Path(os.environ.get("ADVANCED_DATA_SCIENCE_ROOT", Path.cwd())).expanduser().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "Datasets").is_dir() and (candidate / "project_notebooks").is_dir():
        ROOT = candidate
        break
PROJECT_DIRECTORY = ROOT
PROJECT_PATH = ROOT / "project_notebooks/from_linguistic_context_to_semantic_structure"
DATASETS_ROOT = Path(os.environ.get("DATASETS_ROOT", str(ROOT / "Datasets"))).expanduser().resolve()
SOURCE_DIRECTORY = DATASETS_ROOT / "arxiv_research_papers_2020"
EXPERIMENT_DIRECTORY = PROJECT_PATH
RESULTS_DIRECTORY = PROJECT_PATH / "results"
FIGURE_DIRECTORY = RESULTS_DIRECTORY / "figures"
NORMALIZED_TEXT_DIRECTORY = SOURCE_DIRECTORY / "normalized_text"
DATASET_AVAILABLE = False
try:
    required = ["source_manifest.json", "source_text_register.json", "document_metadata.json", "normalization_manifest.json"]
    missing = [name for name in required if not (SOURCE_DIRECTORY / name).is_file()]
    if missing:
        raise FileNotFoundError("Dataset unavailable: " + ", ".join(missing) + ". Run the arXiv section of Datasets/Datasets.ipynb manually, then set DATASETS_ROOT to that saved dataset location.")
    source_manifest = json.loads((SOURCE_DIRECTORY / "source_manifest.json").read_text())
    normalization_manifest = json.loads((SOURCE_DIRECTORY / "normalization_manifest.json").read_text())
    metadata = json.loads((SOURCE_DIRECTORY / "document_metadata.json").read_text())
    DATASET_REVISION = "a68009864dbc3b1decb1a37a6c4f769887169a44"
    assert source_manifest["revision"] == normalization_manifest["source_revision"] == DATASET_REVISION
    normalization_records = normalization_manifest["documents"]
    assert len(normalization_records) == len(source_manifest["documents"]) == 8836
    missing_texts = [r["document_id"] for r in normalization_records if not (NORMALIZED_TEXT_DIRECTORY / (r["document_id"] + ".txt")).is_file()]
    if missing_texts:
        raise FileNotFoundError(f"{len(missing_texts)} dataset texts are absent. Run Datasets/Datasets.ipynb manually; this project does not modify datasets.")
    RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    print(f"Reading {len(normalization_records):,} source documents from {SOURCE_DIRECTORY}")
    DATASET_AVAILABLE = True
except FileNotFoundError:
    print(f"Dataset not found: {SOURCE_DIRECTORY}. Run Datasets/Datasets.ipynb manually, then rerun this cell.")


## Project outputs

Prepare numerical imports, atomic result writing and inline diagnostic figures. These functions write only to this project’s output directories.


In [ ]:
import hashlib, csv, io, time, math
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, Image

def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)

def display_figure(figure, name):
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    (FIGURE_DIRECTORY / name).write_bytes(buffer.getvalue())
    display(Image(data=buffer.getvalue()))
    plt.close(figure)


## Scientific computation libraries

Import the clustering, matrix decomposition and pretrained transformer interfaces required by the numerical and classification stages.

The following cells define the complete implementation. All empirical stages use `corpus/` and source identities preserved above. Large model and result files are retained as verified parts for Git storage and reconstructed byte-for-byte on execution.


In [ ]:
import os, gc, copy, shutil, time, warnings, platform, re
from dataclasses import dataclass
from types import SimpleNamespace
from datetime import datetime, timezone
import scipy
from scipy.sparse.linalg import svds
import sklearn
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import calinski_harabasz_score, accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, log_loss
from threadpoolctl import threadpool_limits
import torch
from torch import nn
from transformers import BertConfig, BertModel, BertTokenizerFast, AutoTokenizer, AutoModelForTokenClassification, pipeline
threadpool_limits(2)
torch.set_num_threads(2)


## Source partitions and resumable computation

Define source-family separation, checkpoint handling, model retrieval and contiguous chunk boundaries. These functions preserve the source population when execution resumes.


In [ ]:
# Purpose: preserve one run configuration, source partitions and restartable stages.
BERT_REPOSITORY = 'google-bert/bert-base-uncased'
BERT_REVISION = '86b5e0934494bd15c9632b12f734a8a67f723594'
SEGMENTER_REPOSITORY = 'mirth/chonky_mmbert_small_multilingual_1'
SEGMENTER_REVISION = '4a92de5574053676cbc5fd06bac86a6e13588000'
CLASSIFIER_SEEDS = (19, 29, 41)
CLASSIFIERS = ('bert_only', 'bert_tfidf', 'bert_profile', 'bert_full',
               'bert_full_svd', 'legacy_frozen_bert_only', 'legacy_frozen_svd_cv')
TOKEN_PATTERN = r'(?u)\b\w\w+\b'
RUN_STARTED = time.monotonic()
RUN_BUDGET = float(os.environ.get('PIPELINE_TIME_BUDGET_SECONDS', '0'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CheckpointPause(RuntimeError):
    """A scheduled pause after durable progress, never a completed experiment."""

def check_time_budget():
    if RUN_BUDGET and time.monotonic() - RUN_STARTED > RUN_BUDGET:
        raise CheckpointPause('Saved checkpoint; resume the same notebook to continue.')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()

def object_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def code_fingerprint():
    notebook = json.loads((PROJECT_PATH / 'Proof.ipynb').read_text())
    return object_digest([(c['id'], c['source']) for c in notebook['cells'] if c['cell_type'] == 'code'])

def save_array_file(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp.npz')
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def save_torch_file(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

def pack_large_files(folder, threshold=90 * 1024**2, part_size=64 * 1024**2):
    """Persist byte-identical large artifacts in Git-compatible parts; keep originals in place."""
    folder = Path(folder)
    paths = [p for p in folder.rglob('*') if p.is_file() and not any(x.endswith('.parts') for x in p.parts)
             and not p.name.endswith('.tmp') and p.stat().st_size >= threshold]
    for path in paths:
        parts = path.with_name(path.name + '.parts')
        parts.mkdir(exist_ok=True)
        digest = file_sha256(path)
        manifest_path = parts / 'manifest.json'
        if manifest_path.exists():
            prior = json.loads(manifest_path.read_text())
            if prior.get('sha256') == digest and all((parts / r['name']).exists() and
                file_sha256(parts / r['name']) == r['sha256'] for r in prior['parts']):
                continue
        records = []
        with path.open('rb') as stream:
            number = 0
            while block := stream.read(part_size):
                name = f'{number:06d}.bin'
                temporary = parts / (name + '.tmp')
                temporary.write_bytes(block)
                temporary.replace(parts / name)
                records.append(dict(name=name, bytes=len(block), sha256=hashlib.sha256(block).hexdigest()))
                number += 1
        save_json(manifest_path, dict(path=path.name, bytes=path.stat().st_size, sha256=digest, parts=records))
        keep = {r['name'] for r in records}
        for old in parts.glob('*.bin'):
            if old.name not in keep:
                old.unlink()

def restore_large_files(folder):
    for manifest_path in sorted(Path(folder).rglob('*.parts/manifest.json')):
        manifest = json.loads(manifest_path.read_text())
        target = manifest_path.parent.parent / manifest['path']
        if target.exists() and file_sha256(target) == manifest['sha256']:
            continue
        temporary = target.with_name(target.name + '.tmp')
        with temporary.open('wb') as stream:
            for record in manifest['parts']:
                block = (manifest_path.parent / record['name']).read_bytes()
                assert len(block) == record['bytes'] and hashlib.sha256(block).hexdigest() == record['sha256']
                stream.write(block)
        assert file_sha256(temporary) == manifest['sha256']
        temporary.replace(target)

def obtain_pretrained_model(repository, revision, folder):
    from huggingface_hub import snapshot_download
    folder = Path(folder)
    manifest_path = folder / 'model_manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        assert manifest['repository'] == repository and manifest['revision'] == revision
        for name, digest in manifest['sha256'].items():
            assert file_sha256(folder / name) == digest, f'Changed pretrained model file: {name}'
        return folder
    snapshot_download(repository, revision=revision, local_dir=folder,
        allow_patterns=['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json',
                        'special_tokens_map.json', 'vocab.txt'], max_workers=2)
    assert (folder / 'model.safetensors').is_file()
    names = sorted(p.name for p in folder.iterdir() if p.is_file())
    save_json(manifest_path, dict(repository=repository, revision=revision,
        source_url=f'https://huggingface.co/{repository}/tree/{revision}',
        sha256={name: file_sha256(folder / name) for name in names}))
    return folder

def source_partitions(records, metadata_by_id):
    """Keep exact normalized duplicates and equal nonempty DOIs in one source family."""
    records = sorted(records, key=lambda r: r['document_id'])
    parents = list(range(len(records)))
    def root(i):
        while parents[i] != i:
            parents[i] = parents[parents[i]]
            i = parents[i]
        return i
    seen = {}
    for i, record in enumerate(records):
        doi = metadata_by_id[record['document_id']].get('doi', '').strip().lower()
        keys = [('normalized', record['normalized_sha256'])]
        if doi:
            keys.append(('doi', doi.removeprefix('https://doi.org/')))
        for key in keys:
            if key in seen:
                parents[root(i)] = root(seen[key])
            else:
                seen[key] = i
    families = np.array([root(i) for i in range(len(records))])
    topics = sorted({metadata_by_id[r['document_id']]['primary_category'] for r in records})
    labels = np.array([topics.index(metadata_by_id[r['document_id']]['primary_category']) for r in records])
    folds = np.empty(len(records), dtype=int)
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=29)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='The least populated class')
        for fold, (_, indices) in enumerate(splitter.split(np.zeros(len(records)), labels, families)):
            folds[indices] = fold
    splits = np.where(folds == 0, 'test', np.where(folds == 1, 'validation', 'train')).astype(object)
    # Source-category support is decided from labels before vocabulary/model fitting.
    # Whole families move together; every document remains. The adjustment is recorded.
    adjustments = []
    for label, topic in enumerate(topics):
        if not np.any((labels == label) & (splits == 'train')):
            candidates = sorted(set(families[labels == label]), key=lambda g: object_digest({'seed':29,'family':int(g)}))
            family = candidates[0]
            indices = np.flatnonzero(families == family)
            adjustments.append(dict(category=topic, family=int(family), prior_split=str(splits[indices[0]]),
                                    document_ids=[records[i]['document_id'] for i in indices]))
            splits[indices] = 'train'
    documents = [dict(id=i, document_id=r['document_id'], label=int(labels[i]), topic=topics[labels[i]],
                      split=str(splits[i]), family=int(families[i]), body_start=0,
                      source_sha256=r['source_sha256'], normalized_sha256=r['normalized_sha256'])
                 for i, r in enumerate(records)]
    assert set(splits) == {'train', 'validation', 'test'}
    assert all(len(set(splits[families == g])) == 1 for g in np.unique(families))
    return dict(documents=documents, topics=topics, training_support_adjustments=adjustments,
        rule='Five-fold StratifiedGroupKFold, seed29; test fold0, validation fold1, training folds2-4; move a whole deterministic family to training only when a category otherwise has no training support.',
        family_rule='Exact normalized-text SHA256 or equal nonempty normalized DOI, transitively connected.',
        class_support={s:{topic:int(np.count_nonzero((labels==label)&(splits==s))) for label,topic in enumerate(topics)}
                       for s in ('train','validation','test')}, source_revision=DATASET_REVISION)

def partition_sources():
    destination = RESULTS_DIRECTORY / 'source_partitions.json'
    identity = object_digest(dict(revision=DATASET_REVISION, records=normalization_records,
        metadata={k:{f:v.get(f) for f in ('primary_category','doi')} for k,v in metadata.items()}, recipe='grouped-arxiv-primary-v1'))
    if destination.exists():
        saved = json.loads(destination.read_text())
        assert saved['identity'] == identity, 'Source partition inputs changed; preserve and reconcile existing results.'
        return saved
    saved = source_partitions(normalization_records, metadata)
    saved['identity'] = identity
    save_json(destination, saved)
    return saved

def validated_chunk_ends(text, predictions):
    """Use Chonky's separator-end slicing; model window edges are not boundaries."""
    from chonky import split_into_semantic_chunks
    ends = [int(p['end']) for p in predictions]
    assert all(0 < end <= len(text) for end in ends)
    assert ends == sorted(set(ends)), 'Separator ends must be strictly increasing.'
    pieces = [piece for piece in split_into_semantic_chunks(text, predictions) if piece]
    if not pieces:
        pieces = ['']
    assert ''.join(pieces) == text
    return np.cumsum([len(piece) for piece in pieces]).tolist()

def segment_sources(partitions, model_folder):
    folder = RESULTS_DIRECTORY / 'segments'
    folder.mkdir(exist_ok=True)
    identity = object_digest(dict(partitions=partitions['identity'], model=SEGMENTER_REVISION,
        package='chonky==0.1.7', max_length=1024, stride=512, aggregation='simple', boundary='separator end'))
    pending = []
    for document in partitions['documents']:
        path = folder / (document['document_id'] + '.json')
        if path.exists():
            saved = json.loads(path.read_text())
            assert saved['identity'] == identity and saved['normalized_sha256'] == document['normalized_sha256']
        else:
            pending.append(document)
    if pending:
        tokenizer = AutoTokenizer.from_pretrained(model_folder, model_max_length=1024, local_files_only=True)
        segmenter = AutoModelForTokenClassification.from_pretrained(model_folder, local_files_only=True,
            attn_implementation='sdpa')
        assert segmenter.config.id2label == {0:'O', 1:'separator'}
        splitter = pipeline('ner', model=segmenter, tokenizer=tokenizer, device=DEVICE,
                            aggregation_strategy='simple', stride=512)
        for number, document in enumerate(pending, 1):
            check_time_budget()
            text = (NORMALIZED_TEXT_DIRECTORY / (document['document_id'] + '.txt')).read_text()
            assert hashlib.sha256(text.encode()).hexdigest() == document['normalized_sha256']
            started = time.monotonic()
            predictions = splitter(text) if text.strip() else []
            ends = validated_chunk_ends(text, predictions)
            save_json(folder / (document['document_id'] + '.json'), dict(identity=identity,
                document_id=document['document_id'], normalized_sha256=document['normalized_sha256'], ends=ends,
                separators=[dict(start=int(p['start']),end=int(p['end']),score=float(p['score'])) for p in predictions],
                seconds=time.monotonic()-started))
            print(f"Contextual chunking: {len(partitions['documents'])-len(pending)+number:,}/{len(partitions['documents']):,} sources", flush=True)
        del splitter, segmenter, tokenizer
        gc.collect()
    lengths = []
    snippets = 0
    for document in partitions['documents']:
        text = (NORMALIZED_TEXT_DIRECTORY / (document['document_id'] + '.txt')).read_text()
        record = json.loads((folder / (document['document_id'] + '.json')).read_text())
        ends = record['ends']
        assert ends and ends[-1] == len(text)
        assert (ends == [0] and not text) or all(a < b for a,b in zip([0]+ends[:-1],ends))
        snippets += len(ends)
        lengths.extend(b-a for a,b in zip([0]+ends[:-1], ends))
    report = dict(complete=True, identity=identity, documents=len(partitions['documents']), snippets=snippets,
        source_revision=DATASET_REVISION, model_revision=SEGMENTER_REVISION,
        boundary_rule='Chonky split_into_semantic_chunks: split at each predicted separator end, then retain the tail.',
        checks=dict(all_sources_retained=True, exact_contiguous_coverage=True, no_fixed_length_final_chunks=True),
        character_length_quantiles=np.quantile(lengths,[0,.25,.5,.75,.9,.99,1]).tolist())
    save_json(RESULTS_DIRECTORY / 'segmentation.json', report)
    figure, ax = plt.subplots(figsize=(7,4))
    ax.hist(lengths, bins=60); ax.set(xlabel='Characters per contextual chunk',ylabel='Chunks',title=f"{snippets:,} chunks from {len(partitions['documents']):,} sources")
    display_figure(figure, 'contextual_chunk_lengths.png')
    return report

def load_corpus():
    partitions = json.loads((RESULTS_DIRECTORY / 'source_partitions.json').read_text())
    corpus = dict(documents=[], snippets=[], topics=partitions['topics'])
    for document in partitions['documents']:
        document = dict(document)
        text = (NORMALIZED_TEXT_DIRECTORY / (document['document_id']+'.txt')).read_text()
        document['text'] = text
        corpus['documents'].append(document)
        ends = json.loads((RESULTS_DIRECTORY / 'segments' / (document['document_id']+'.json')).read_text())['ends']
        for start,end in zip([0]+ends[:-1],ends):
            corpus['snippets'].append(dict(doc=document['id'],split=document['split'],label=document['label'],
                                          start=start,end=end,text=text[start:end]))
    validate_corpus(corpus)
    target_path = RESULTS_DIRECTORY / 'targets.json'
    if target_path.exists():
        corpus['targets'] = json.loads(target_path.read_text())['targets']
    return corpus


## Grouped moments and feature-aware classification

Define the original distance statistics and classifier interface. Statistical features join the target occurrence representation in the trainable classification head.


In [ ]:
# Purpose: original numerical definitions and the original supplied-BERT training interface.
def regular_idf(counts):
    counts = np.asarray(counts, dtype=float)
    if counts.ndim != 2 or min(counts.shape) == 0:
        raise ValueError('Counts must be a nonempty observation-by-vocabulary matrix.')
    if not np.isfinite(counts).all() or (counts < 0).any():
        raise ValueError('Counts must be finite and nonnegative.')
    df = (counts > 0).sum(axis=0)
    if (df == 0).any():
        raise ValueError('Reference vocabulary must be observed in the reference corpus.')
    return np.log(counts.shape[0] / df)

def l2_normalize_rows(weights):
    """Unit L2 length, scaling before squaring to preserve tiny finite rows.

    The returned original-unit norm can exceed float64's range. Normalized
    coordinates are computed independently of that metadata overflow.
    """
    weights = np.asarray(weights, dtype=float)
    if weights.ndim != 2 or not np.isfinite(weights).all():
        raise ValueError('Passage weights must be a finite two-dimensional array.')
    scale = np.max(np.abs(weights), axis=1, initial=0)
    scaled = np.zeros_like(weights)
    np.divide(weights, scale[:, None], out=scaled, where=scale[:, None] > 0)
    scaled_norms = np.linalg.norm(scaled, axis=1)
    normalized = np.zeros_like(weights)
    np.divide(scaled, scaled_norms[:, None], out=normalized,
              where=scaled_norms[:, None] > 0)
    with np.errstate(over='ignore'):
        norms = scale * scaled_norms
    return normalized, norms

def aggregate_distance_groups(distances, cluster_ids):
    """Exactly combine distance moments across groups, separately on each axis.

    Input measurements are distances only. Group means here are means of
    distances, not TF-IDF values or contextual-centroid coordinates. CV is
    formed from column-scaled centered moments, avoiding raw-square underflow
    and cancellation. Original-unit moments are retained as metadata, with
    representability masks. Scaled moments remain available when an original
    squared moment lies outside float64's range.
    """
    distances = np.asarray(distances, dtype=float)
    cluster_ids = np.asarray(cluster_ids)
    if distances.ndim != 2 or min(distances.shape, default=0) == 0 or cluster_ids.shape != (len(distances),):
        raise ValueError('Distances must have positive row and vocabulary widths, with one cluster identifier per row.')
    if cluster_ids.dtype.kind not in 'iu':
        raise ValueError('Cluster identifiers must be integers and retain their identities in group summaries.')
    if not np.isfinite(distances).all() or (distances < 0).any():
        raise ValueError('Coordinate distances must be finite and nonnegative.')
    scale = np.max(distances, axis=0)
    scaled = np.zeros_like(distances)
    np.divide(distances, scale, out=scaled, where=scale > 0)
    summaries=[]
    for g in np.unique(cluster_ids):
        block=scaled[cluster_ids==g]
        mean=block.mean(axis=0)
        total=block.sum(axis=0)
        square_total=(block**2).sum(axis=0)
        centered_total=((block-mean)**2).sum(axis=0)
        with np.errstate(over='ignore', under='ignore'):
            summaries.append(dict(cluster=int(g),count=len(block),sum=scale*total,
                sum_squares=scale*(scale*square_total),mean=scale*mean,
                centered_sum_squares=scale*(scale*centered_total),
                scaled_sum=total,scaled_mean=mean,
                scaled_sum_squares=square_total,
                scaled_centered_sum_squares=centered_total))
    n=sum(group['count'] for group in summaries)
    scaled_mean=sum(group['scaled_sum'] for group in summaries)/n
    # Stable counterpart of Q/n-(S/n)^2, retaining between-group mean shifts.
    scaled_variance=sum(group['scaled_centered_sum_squares']+
        group['count']*(group['scaled_mean']-scaled_mean)**2
        for group in summaries)/n
    valid=scale > 0
    score=np.full_like(scaled_mean,np.nan)
    np.divide(np.sqrt(scaled_variance),scaled_mean,out=score,where=valid)
    with np.errstate(over='ignore', under='ignore'):
        mean=scale*scaled_mean
        standard_deviation=scale*np.sqrt(scaled_variance)
        variance=standard_deviation**2
    mean_representable=np.isfinite(mean) & ((mean > 0) | ~valid)
    variance_representable=np.isfinite(variance) & (
        (variance > 0) | (scaled_variance == 0))
    return dict(mean=mean,variance=variance,cv=score,valid=valid,groups=summaries,
        distance_scale=scale,scaled_mean=scaled_mean,scaled_variance=scaled_variance,
        mean_representable=mean_representable,
        variance_representable=variance_representable)

def coordinate_distance_cv(weights, centroids):
    """Choose a full-vector nearest centroid, then aggregate each axis over rows."""
    weights = np.asarray(weights, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    if weights.ndim != 2 or centroids.ndim != 2 or min(weights.shape, default=0) == 0 or min(centroids.shape, default=0) == 0:
        raise ValueError('Weights and centroids must be nonempty two-dimensional arrays.')
    if weights.shape[1] != centroids.shape[1]:
        raise ValueError('Passage and centroid vocabulary dimensions must agree.')
    if not np.isfinite(weights).all() or not np.isfinite(centroids).all():
        raise ValueError('Coordinates must be finite.')
    # Bounded by one row-to-centroid block at a time, not an n*k*p tensor.
    assignment = np.empty(len(weights), dtype=np.int64)
    distances = np.empty_like(weights)
    for i, row in enumerate(weights):
        lengths = np.hypot.reduce(centroids-row, axis=1)
        assignment[i] = int(np.argmin(lengths))  # Stable first-index tie rule.
        distances[i] = np.abs(row-centroids[assignment[i]])
    aggregated=aggregate_distance_groups(distances,assignment)
    return dict(assignment=assignment,distances=distances,**aggregated)

def centroid_mean(block):
    """Arithmetic mean preserving exactly constant normalized coordinates.

    Subtracting a member first avoids inventing residuals when repeated,
    non-binary floating-point coordinates would be rounded by sum/count.
    The selected representation bounds all coordinates between zero and one.
    """
    block = np.asarray(block, dtype=float)
    if block.ndim != 2 or not len(block) or not np.isfinite(block).all():
        raise ValueError('A centroid requires a nonempty finite matrix.')
    anchor = block[0]
    return anchor + (block-anchor).mean(axis=0)

@dataclass
class ContextDiagnostic:
    """Euclidean k-means reference selected by maximum finite variance ratio."""
    idf: np.ndarray
    weights: np.ndarray
    centroids: np.ndarray
    cluster_labels: np.ndarray
    word_cv: np.ndarray
    cv_valid: np.ndarray
    selected_group_count: int
    variance_ratio: float
    candidate_results: tuple
    counts: np.ndarray
    row_norms: np.ndarray

    @classmethod
    def fit(cls, counts, group_counts=None, random_state=0, n_init=20):
        """Run Euclidean k-means for each k, then maximize Caliński-Harabasz.

        With no explicit candidate list, examine every k from 2 to n-1.
        A declared smaller list bounds the search on a large reference corpus.
        The score uses the complete reference matrix, not a random minibatch.
        """
        from sklearn.cluster import KMeans
        counts = np.asarray(counts, dtype=float)
        idf = regular_idf(counts)
        weights, row_norms = l2_normalize_rows(counts*idf)
        n = len(weights)
        if group_counts is None:
            group_counts = range(2, n)
        group_counts = list(group_counts)
        if any(int(k) != k or not 2 <= int(k) < n for k in group_counts):
            raise ValueError('Candidate counts must be integers from 2 to n-1.')
        candidates = sorted({int(k) for k in group_counts})
        best_labels = np.zeros(n, dtype=np.int64)
        best_score = float('nan')
        trace = []
        if candidates and np.any(np.ptp(weights, axis=0) > 0):
            distinct = len(np.unique(weights, axis=0))
            for k in candidates:
                if k > distinct:
                    trace.append(dict(k=k, status='insufficient_distinct_rows', score=None))
                    continue
                if k == distinct:
                    trace.append(dict(k=k, status='zero_within_dispersion', score=None,
                                      within=0.0))
                    continue
                model = KMeans(n_clusters=k, init='k-means++', n_init=n_init,
                               random_state=random_state, algorithm='lloyd',
                               tol=0.0, max_iter=1000).fit(weights)
                labels = model.labels_.astype(np.int64)
                # Finish the same Lloyd assignment/mean updates. This makes the
                # centroid and nearest-assignment definitions agree, including
                # the deterministic first-index tie rule used in the diagnostic.
                for _ in range(1000):
                    if len(np.unique(labels)) != k:
                        raise RuntimeError('A k-means candidate has an empty cluster.')
                    centers = np.stack([centroid_mean(weights[labels == j]) for j in range(k)])
                    nearest = coordinate_distance_cv(weights, centers)['assignment']
                    if np.array_equal(labels, nearest):
                        break
                    labels = nearest
                else:
                    raise RuntimeError('Lloyd assignment/centroid consistency did not converge.')
                within = ((weights-centers[labels])**2).sum()
                if within <= 0:
                    trace.append(dict(k=k, status='zero_within_dispersion', score=None))
                    continue  # The variance ratio requires a positive denominator.
                overall = centroid_mean(weights)
                between = sum(np.count_nonzero(labels == j)*
                              np.sum((centers[j]-overall)**2) for j in range(k))
                score = float((between/(k-1))/(within/(n-k)))
                trace.append(dict(k=k, status='evaluated', score=score,
                                  within=float(within), between=float(between)))
                if np.isfinite(score) and (not np.isfinite(best_score) or score > best_score):
                    best_labels, best_score = labels.copy(), score
        k = int(best_labels.max())+1
        centroids = np.stack([centroid_mean(weights[best_labels == j]) for j in range(k)])
        result = coordinate_distance_cv(weights, centroids)
        return cls(idf, weights, centroids, best_labels, result['cv'],
                   result['valid'], k, best_score, tuple(trace), counts.copy(), row_norms)

    def passage_features(self, passage_counts):
        """Full passage rows and categorical cluster ids, with no target distance input."""
        counts = np.asarray(passage_counts, dtype=float)
        if counts.ndim != 2 or counts.shape[1] != len(self.idf):
            raise ValueError('Passage rows must use the fitted vocabulary.')
        if not np.isfinite(counts).all() or (counts < 0).any():
            raise ValueError('Passage counts must be finite and nonnegative.')
        weights, row_norms = l2_normalize_rows(counts*self.idf)
        assignment = coordinate_distance_cv(weights, self.centroids)['assignment']
        return dict(cluster_ids=assignment, passage_tfidf=weights,
                    row_available=row_norms > 0)

    def long_records(self, vocabulary=None):
        """Lossless wide-to-long records defining the primary bipartite multigraph.

        source/target are the two endpoint columns. distance is a separation
        attribute, not an affinity. Zero weights and distances remain present.
        """
        n, p = self.weights.shape
        vocabulary = list(map(str, range(p))) if vocabulary is None else list(vocabulary)
        if len(vocabulary) != p or len(set(vocabulary)) != p:
            raise ValueError('One distinct vocabulary label is required per axis.')
        result = coordinate_distance_cv(self.weights, self.centroids)
        records = []
        for i in range(n):
            g = int(result['assignment'][i])
            for t in range(p):
                records.append(dict(passage=i, word=vocabulary[t], cluster=g,
                    source='word:'+vocabulary[t], target='cluster:'+str(g),
                    tfidf=float(self.weights[i,t]), centroid=float(self.centroids[g,t]),
                    raw_count=float(self.counts[i,t]),
                    unnormalized_tfidf=float(self.counts[i,t]*self.idf[t]),
                    row_norm=float(self.row_norms[i]), row_available=bool(self.row_norms[i] > 0),
                    distance=float(result['distances'][i,t]),
                    word_cv=float(self.word_cv[t]) if self.cv_valid[t] else None,
                    cv_available=bool(self.cv_valid[t])))
        return records

STATISTICS = ('mean','within_ss','prevalence','positive_mean','positive_variance','skewness','excess_kurtosis','count')

def corpus_profiles(weights, cluster_ids):
    """Return word x cluster x statistic values and explicit validity masks."""
    x = np.asarray(weights, dtype=np.float64)
    a = np.asarray(cluster_ids)
    if x.ndim != 2 or min(x.shape) == 0 or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('Finite nonnegative passage-by-word weights required.')
    if a.shape != (len(x),) or not np.issubdtype(a.dtype, np.integer):
        raise ValueError('One integer cluster index per passage required.')
    labels = np.unique(a)
    if not np.array_equal(labels, np.arange(len(labels))):
        raise ValueError('Cluster indices must be contiguous from zero.')
    p, k = x.shape[1], len(labels)
    values = np.zeros((p, k, len(STATISTICS)))
    valid = np.zeros_like(values, dtype=bool)
    means = np.empty((k, p))
    between = np.zeros(p)
    overall = x.mean(axis=0)
    for g in labels:
        block = x[a == g]
        mu = block.mean(axis=0)
        means[g] = mu
        centered = block - mu
        m2 = (centered**2).mean(axis=0)
        values[:, g, 0] = mu
        values[:, g, 1] = (centered**2).sum(axis=0)
        values[:, g, 2] = (block > 0).mean(axis=0)
        values[:, g, 7] = len(block)
        valid[:, g, [0, 1, 2, 7]] = True
        between += len(block) * (mu - overall)**2
        for t in range(p):
            positive = block[:, t][block[:, t] > 0]
            if len(positive):
                values[t, g, 3:5] = (positive.mean(), positive.var(ddof=0))
                valid[t, g, 3:5] = True
            if m2[t] > 0:
                values[t, g, 5] = np.mean(centered[:, t]**3) / m2[t]**1.5
                values[t, g, 6] = np.mean(centered[:, t]**4) / m2[t]**2 - 3
                valid[t, g, 5:7] = True
    if not np.isfinite(values).all():
        raise ValueError('Profile overflow: rescale weights before estimation.')
    return dict(values=values, valid=valid, means=means, between=between,
                statistics=STATISTICS)

def occurrence_features(profiles, passage_weights, target_word_ids):
    """Concatenate TF-IDF, full profile, validity, between SS and local distance.

    Output width is 1 + 2*k*s + 2. Negative/OOV vocabulary identifiers are
    rejected: extraction must explicitly handle missing vocabulary coverage.
    """
    x = np.asarray(passage_weights, dtype=np.float64)
    t = np.asarray(target_word_ids)
    values, valid = profiles['values'], profiles['valid']
    p, k, s = values.shape
    if x.ndim != 2 or x.shape[1] != p or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('Finite rows aligned with the fitted vocabulary required.')
    if t.shape != (len(x),) or not np.issubdtype(t.dtype, np.integer) or (t < 0).any() or (t >= p).any():
        raise ValueError('Each target occurrence needs a known vocabulary index.')
    a = np.square(x[:, None, :] - profiles['means'][None, :, :]).sum(-1).argmin(1)
    w = x[np.arange(len(x)), t]
    delta = np.abs(w - profiles['means'][a, t])
    return np.concatenate((w[:, None], values[t].reshape(len(x), k*s),
                           valid[t].reshape(len(x), k*s).astype(float),
                           profiles['between'][t, None], delta[:, None]), axis=1)

class ContextualBehaviorBert(nn.Module):
    """A supplied BertModel plus a trainable head and trainable final block."""
    def __init__(self, bert, feature_width, number_of_labels, hidden_width=128):
        super().__init__()
        if feature_width < 0 or number_of_labels < 2:
            raise ValueError('Nonnegative feature width and at least two labels required.')
        if bert.config.model_type != 'bert' or not hasattr(bert.encoder, 'layer'):
            raise ValueError('An unmodified BertModel is required.')
        self.bert = bert
        self.feature_width = feature_width
        self.bert.requires_grad_(False)
        self.bert.encoder.layer[-1].requires_grad_(True)
        self.head = nn.Sequential(nn.Linear(bert.config.hidden_size + feature_width, hidden_width),
                                  nn.Tanh(), nn.Linear(hidden_width, number_of_labels))
        self.train(True)

    def train(self, mode=True):
        super().train(mode)
        self.bert.eval()
        self.bert.encoder.layer[-1].train(mode)
        return self

    def forward(self, input_ids, attention_mask, target_mask, features,
                labels=None, token_type_ids=None):
        if input_ids.ndim != 2 or input_ids.shape[1] > self.bert.config.max_position_embeddings:
            raise ValueError('Token batch exceeds the supplied BERT interface.')
        if attention_mask.shape != input_ids.shape or target_mask.shape != input_ids.shape:
            raise ValueError('Attention and occurrence masks must align with token IDs.')
        if not ((target_mask == 0) | (target_mask == 1)).all() or not ((attention_mask == 0) | (attention_mask == 1)).all():
            raise ValueError('Masks must be binary.')
        if (target_mask > attention_mask).any() or (target_mask.sum(1) == 0).any():
            raise ValueError('Target occurrence must contain attended WordPieces.')
        if features.shape != (len(input_ids), self.feature_width) or not torch.isfinite(features).all():
            raise ValueError('Aligned finite statistical features required.')
        states = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids).last_hidden_state
        mask = target_mask.to(states.dtype).unsqueeze(-1)
        h = (states * mask).sum(1) / mask.sum(1)
        logits = self.head(torch.cat((h, features.to(h.dtype)), dim=1))
        loss = None if labels is None else nn.functional.cross_entropy(logits, labels)
        return dict(logits=logits, loss=loss)


## Sparse numerical operations

Define training-only term weighting, Euclidean clustering, grouped statistics and subsequent singular value decomposition using sparse storage where applicable.


In [ ]:
# Purpose: bounded sparse storage of the original mathematical operations.
def bounded_regular_idf(counts, width=256):
    """Original regular_idf applied independently to disjoint word columns."""
    counts = counts.tocsc()
    out = np.empty(counts.shape[1], dtype=np.float64)
    for start in range(0, counts.shape[1], width):
        stop = min(start + width, counts.shape[1])
        out[start:stop] = regular_idf(counts[:, start:stop].toarray())
    return out

def bounded_tfidf(counts, idf, memory_mb=128):
    """Original stable L2 normalization on complete rows, in bounded batches."""
    p = counts.shape[1]
    rows_per_batch = max(1, int(memory_mb * 1024**2 // (8 * max(1, p) * 6)))
    blocks = []
    norms = np.empty(counts.shape[0], dtype=np.float64)
    for start in range(0, counts.shape[0], rows_per_batch):
        stop = min(start + rows_per_batch, counts.shape[0])
        dense = counts[start:stop].toarray().astype(np.float64, copy=False)
        values, n = l2_normalize_rows(dense * idf)
        blocks.append(sparse.csr_matrix(values))
        norms[start:stop] = n
    result = sparse.vstack(blocks, format='csr')
    result.sort_indices()
    return result, norms

def bounded_centers(x, labels, k, width=256):
    """Use the original anchor-centered mean; batching does not split rows."""
    p = x.shape[1]
    centers = np.empty((k, p), dtype=np.float64)
    membership = [np.flatnonzero(labels == g) for g in range(k)]
    if any(len(i) == 0 for i in membership):
        raise RuntimeError('A fitted k-means cluster is empty.')
    for g, ii in enumerate(membership):
        block = x[ii].tocsc()
        for start in range(0, p, width):
            stop = min(start + width, p)
            centers[g, start:stop] = centroid_mean(block[:, start:stop].toarray())
    return centers

def nearest_euclidean(x, centers, batch_size=512):
    """Full-coordinate Euclidean assignment, with original hypot for near ties.

    The squared-norm identity gives every center's full L2 distance. Values
    within 1e-10 of the minimum are recomputed by the original stable hypot
    reduction, retaining the original first-index tie convention. Normalized
    rows/centroids have norm <=1; 1e-10 is deliberately conservative compared
    with float64 dot-product roundoff for these corpus matrices.
    """
    if not np.isfinite(centers).all():
        raise ValueError('Nonfinite centers.')
    cn = np.einsum('ij,ij->i', centers, centers)
    out = np.empty(x.shape[0], dtype=np.int64)
    for start in range(0, x.shape[0], batch_size):
        stop = min(start + batch_size, x.shape[0])
        block = x[start:stop]
        xn = np.asarray(block.multiply(block).sum(axis=1)).ravel()
        distances2 = xn[:, None] + cn[None, :] - 2 * (block @ centers.T)
        minimum = distances2.min(axis=1)
        result = distances2.argmin(axis=1)
        close = distances2 <= minimum[:, None] + 1e-10
        for local in np.flatnonzero(close.sum(axis=1) > 1):
            candidates = np.flatnonzero(close[local])
            row = block[local].toarray().ravel()
            d = np.hypot.reduce(centers[candidates] - row, axis=1)
            result[local] = candidates[int(d.argmin())]
        out[start:stop] = result
    return out

def bounded_sums_of_squares(x, centers, labels, width=256):
    within = 0.0
    between = 0.0
    n = np.bincount(labels, minlength=len(centers))
    xc = x.tocsc()
    for start in range(0, x.shape[1], width):
        stop = min(start + width, x.shape[1])
        block = xc[:, start:stop].toarray()
        c = centers[:, start:stop]
        within += float(np.square(block - c[labels]).sum())
        mu = centroid_mean(block)
        between += float((n[:, None] * np.square(c - mu)).sum())
    return within, between

def fit_candidate(x, k, n_init=20, seed=29, dense_fit=None, fit_storage='csr', initial_centers=None):
    # Explicit storage mode: the objective, original vocabulary, full Lloyd
    # updates, multistart count and finishing rules remain identical. Random
    # initialization arithmetic can nevertheless yield different local optima.
    if fit_storage == 'dense':
        fit_input = x.toarray() if dense_fit is None else dense_fit
    elif fit_storage == 'csr':
        fit_input = x.tocsr()
    else:
        raise ValueError('Fitting storage must be dense or csr.')
    init = 'k-means++' if initial_centers is None else initial_centers
    if initial_centers is not None and n_init != 1:
        raise ValueError('Explicit centers are reserved for a single-init sensitivity check.')
    model = KMeans(n_clusters=k, init='k-means++', n_init=n_init,
                   random_state=seed, algorithm='lloyd', tol=0.0,
                   max_iter=1000, copy_x=True)
    model.set_params(init=init)
    model.fit(fit_input)
    labels = model.labels_.astype(np.int64)
    for finish_iteration in range(1000):
        centers = bounded_centers(x, labels, k)
        nearest = nearest_euclidean(x, centers)
        if np.array_equal(nearest, labels):
            break
        labels = nearest
    else:
        raise RuntimeError('Assignment/centroid consistency did not converge.')
    within, between = bounded_sums_of_squares(x, centers, labels)
    score = float((between / (k - 1)) / (within / (x.shape[0] - k))) if within > 0 else None
    status = 'evaluated' if score is not None else 'zero_within_dispersion'
    if score is not None and not np.isfinite(score):
        score = None
        status = 'nonfinite_variance_ratio'
    counts = np.bincount(labels, minlength=k)
    result = dict(k=int(k), status=status,
                  CH=score, within_ss=within, between_ss=between,
                  smallest_cluster=int(counts.min()), largest_cluster=int(counts.max()),
                  singletons=int((counts == 1).sum()), mean_cluster_size=float(counts.mean()),
                  lloyd_iterations=int(model.n_iter_), consistency_updates=finish_iteration,
                  cluster_sizes=counts.tolist())
    result['fit_storage'] = fit_storage
    return result, centers, labels

def cv_and_profiles(x, centers, labels, width=128):
    """Original grouped-CV and corpus_profiles on disjoint vocabulary blocks."""
    n, p = x.shape
    k = len(centers)
    cv = np.full(p, np.nan)
    valid = np.zeros(p, bool)
    distance_mean = np.empty(p)
    distance_variance = np.empty(p)
    group_mean = np.empty((k, p))
    group_centered_ss = np.empty((k, p))
    values = np.empty((p, k, len(STATISTICS)))
    masks = np.empty_like(values, bool)
    means = np.empty((k, p))
    between = np.empty(p)
    direct_error = 0.0
    xc = x.tocsc()
    for start in range(0, p, width):
        stop = min(start + width, p)
        block = xc[:, start:stop].toarray()
        distances = np.abs(block - centers[labels, start:stop])
        agg = aggregate_distance_groups(distances, labels)
        cv[start:stop] = agg['cv']
        valid[start:stop] = agg['valid']
        distance_mean[start:stop] = agg['mean']
        distance_variance[start:stop] = agg['variance']
        for g in agg['groups']:
            group_mean[g['cluster'], start:stop] = g['mean']
            group_centered_ss[g['cluster'], start:stop] = g['centered_sum_squares']
        available = distances.mean(axis=0) > 0
        direct = distances[:, available].std(axis=0, ddof=0) / distances[:, available].mean(axis=0)
        np.testing.assert_array_equal(available, agg['valid'])
        np.testing.assert_allclose(direct, agg['cv'][available], rtol=1e-10, atol=1e-10)
        if direct.size:
            direct_error = max(direct_error, float(np.max(np.abs(direct - agg['cv'][available]))))
        profile = corpus_profiles(block, labels)
        values[start:stop] = profile['values']
        masks[start:stop] = profile['valid']
        means[:, start:stop] = profile['means']
        between[start:stop] = profile['between']
    np.testing.assert_allclose(means, centers, rtol=1e-12, atol=1e-14)
    return dict(cv=cv, cv_valid=valid, cv_distance_mean=distance_mean,
                cv_distance_variance=distance_variance,
                cv_group_mean=group_mean, cv_group_centered_ss=group_centered_ss,
                cv_group_counts=np.bincount(labels, minlength=k),
                profile_values=values, profile_valid=masks,
                profile_means=means, profile_between=between), direct_error

def target_records(corpus, x, vocabulary):
    """Select source spans from already fixed TF-IDF rows without using labels."""
    word_ids = np.full(x.shape[0], -1, dtype=np.int64)
    targets = []
    for i, s in enumerate(corpus['snippets']):
        row = x[i]
        record = dict(snippet=i, doc=s['doc'], split=s['split'], label=s['label'])
        if row.nnz and row.data.max(initial=0) > 0:
            maximum = row.data.max()
            wordid = int(row.indices[row.data == maximum].min())
            word = str(vocabulary[wordid])
            match = next((m for m in re.finditer(TOKEN_PATTERN, s['text'])
                          if m.group().lower() == word), None)
            if match is None:
                raise AssertionError(f'Vocabulary/source occurrence mismatch for snippet {i}.')
            word_ids[i] = wordid
            record.update(word=word, word_id=wordid, start=match.start(), end=match.end(),
                          has_target=True, reason=None)
        else:
            record.update(word=None, word_id=-1, start=0, end=0, has_target=False,
                          reason='no_positive_training_vocabulary_weight')
        targets.append(record)
    return targets, word_ids

def make_targets(corpus, x, vocabulary, profiles, centers, assignments):
    k, p = centers.shape
    features = np.zeros((x.shape[0], 1 + 2 * k * len(STATISTICS) + 2))
    targets, word_ids = target_records(corpus, x, vocabulary)
    # corpus_profiles uses an arithmetic mean, as specified in its original code.
    # Preserve its original nearest-mean feature readout separately from k-means
    # memberships used for CV; these are numerically checked to agree below.
    feature_assignment = nearest_euclidean(x, profiles['profile_means'])
    mismatches = int(np.count_nonzero(feature_assignment != assignments))
    if mismatches:
        raise AssertionError('Occurrence readout must preserve the fixed full-vector assignments.')
    for i, wordid in enumerate(word_ids):
        if wordid < 0:
            continue
        w = float(x[i, wordid])
        g = feature_assignment[i]
        features[i] = np.concatenate((
            [w], profiles['profile_values'][wordid].ravel(),
            profiles['profile_valid'][wordid].ravel().astype(float),
            [profiles['profile_between'][wordid],
             abs(w - profiles['profile_means'][g, wordid])]))
    if not np.isfinite(features).all():
        raise AssertionError('Occurrence features must be finite.')
    return targets, features, word_ids, mismatches

def leading_svd(train, rank, seed=29):
    """Leading singular triplets, with no randomized low-rank approximation.

    SciPy 1.17.0 svds documents which='LM' as the largest singular triplets
    and tol=0 as machine precision. ARPACK uses the smaller normal operator.
    Full retained rank requires the original dense NumPy solver because
    ARPACK requires k < min(shape). Zero matrices admit any orthonormal
    bases; the leading coordinate bases satisfy both triplet equations.
    """
    rank = min(int(rank), *train.shape)
    if not 1 <= rank <= min(train.shape):
        raise ValueError('Retained rank must be positive and within matrix dimensions.')
    if not train.nnz or not np.any(train.data):
        u = np.eye(train.shape[0], rank)
        singular = np.zeros(rank)
        vt = np.eye(rank, train.shape[1])
        solver = 'zero-matrix orthonormal bases'
    elif rank == min(train.shape):
        u, singular, vt = np.linalg.svd(train.toarray(), full_matrices=False)
        u, singular, vt = u[:, :rank], singular[:rank], vt[:rank]
        solver = 'numpy.linalg.svd'
    else:
        u, singular, vt = svds(train, k=rank, which='LM', tol=0,
                               solver='arpack', rng=seed,
                               maxiter=max(1000, 10 * min(train.shape)))
        order = np.argsort(singular)[::-1]
        u, singular, vt = u[:, order], singular[order], vt[order]
        solver = 'scipy.sparse.linalg.svds(arpack,which=LM,tol=0)'
    # Fix paired signs for reproducible serialization without changing subspace.
    maxima = np.argmax(np.abs(vt), axis=1)
    signs = np.sign(vt[np.arange(rank), maxima])
    signs[signs == 0] = 1
    vt *= signs[:, None]
    u *= signs[None, :]
    np.testing.assert_allclose(vt @ vt.T, np.eye(rank), rtol=1e-9, atol=1e-9)
    np.testing.assert_allclose(u.T @ u, np.eye(rank), rtol=1e-9, atol=1e-9)
    av_residual = np.linalg.norm(train @ vt.T - u * singular, axis=0)
    atu_residual = np.linalg.norm(train.T @ u - vt.T * singular, axis=0)
    scale = max(float(singular[0]), np.finfo(float).tiny)
    relative = np.maximum(av_residual, atu_residual) / scale
    if not np.isfinite(relative).all() or np.any(relative > 1e-8):
        raise AssertionError('Leading SVD triplet residual exceeds predeclared relative tolerance 1e-8.')
    if not np.all(singular[:-1] >= singular[1:]) or (singular < 0).any():
        raise AssertionError('Singular values must be nonnegative and descending.')
    check = dict(solver=solver, requested_rank=rank, maximum_relative_triplet_residual=float(relative.max()),
                 relative_triplet_tolerance=1e-8, orthogonality_tolerance=1e-9,
                 largest_singular_value=float(singular[0]), smallest_retained_singular_value=float(singular[-1]))
    return u, singular, vt, check

def svd_views(x_train, x_all, cv, rank=32, seed=29):
    """Post-CV: original raw and CV-weighted leading singular subspaces."""
    rank = min(rank, *x_train.shape)
    scores = np.nan_to_num(cv, nan=0.0)
    result = {}
    checks = {}
    for name, train, all_rows in [('raw', x_train, x_all),
                                  ('cv', x_train.multiply(scores).tocsr(),
                                   x_all.multiply(scores).tocsr())]:
        _, singular, vt, checked = leading_svd(train, rank, seed)
        basis = vt.T
        np.testing.assert_allclose(basis.T @ basis, np.eye(rank), rtol=1e-9, atol=1e-9)
        projection = np.asarray(all_rows @ basis)
        if not np.isfinite(projection).all():
            raise AssertionError('SVD projection contains nonfinite values.')
        result[name + '_svd'] = projection
        result[name + '_basis'] = basis
        result[name + '_singular_values'] = singular
        checks[name] = checked
    result['behavior_basis'] = result['cv_basis']
    return result, checks

def validate_corpus(corpus):
    docs, snippets = corpus['documents'], corpus['snippets']
    if [d['id'] for d in docs] != list(range(len(docs))):
        raise ValueError('Document identifiers must be contiguous and match array order.')
    if not snippets:
        raise ValueError('No snippets supplied.')
    by_document = [[] for _ in docs]
    families = {}
    for d in docs:
        family = d.get('family')
        if family is not None:
            key = str(family)
            if key in families and families[key] != d['split']:
                raise AssertionError('A declared source family crosses data splits.')
            families[key] = d['split']
    for s in snippets:
        d = docs[s['doc']]
        if s['split'] != d['split'] or s['label'] != d['label']:
            raise AssertionError('Snippet labels or splits disagree with their source.')
        if d['text'][s['start']:s['end']] != s['text']:
            raise AssertionError('Snippet offsets do not reconstruct source substring.')
        by_document[d['id']].append(s)
    for d, parts in zip(docs, by_document):
        if not parts:
            raise AssertionError('Every document requires a processing row, even when empty.')
        body_start = d.get('body_start', d.get('body_offset', 0))
        ordered = sorted(parts, key=lambda s: s['start'])
        if ''.join(s['text'] for s in ordered) != d['text'][body_start:]:
            raise AssertionError('Processing rows do not exactly reconstruct the source body.')
        cursor = body_start
        for s in ordered:
            if s['start'] != cursor:
                raise AssertionError('Noncontiguous or overlapping processing rows.')
            cursor = s['end']

def numerical_self_test():
    rng = np.random.default_rng(817)
    counts = rng.poisson(.8, size=(45, 13)).astype(float)
    counts[0] = 0
    counts[1:4] = counts[4]
    counts[:, -1] = 2  # zero-IDF axis must remain present
    c = sparse.csr_matrix(counts)
    idf = bounded_regular_idf(c, width=3)
    np.testing.assert_array_equal(idf, regular_idf(counts))
    x, norms = bounded_tfidf(c, idf, memory_mb=.001)
    dense, dn = l2_normalize_rows(counts * idf)
    np.testing.assert_array_equal(x.toarray(), dense)
    np.testing.assert_array_equal(norms, dn)
    labels = np.repeat(np.arange(3), 15).astype(np.int64)
    centers = bounded_centers(x, labels, 3, width=3)
    original_centers = np.stack([centroid_mean(dense[labels == g]) for g in range(3)])
    np.testing.assert_allclose(centers, original_centers, rtol=1e-14, atol=1e-15)
    np.testing.assert_array_equal(nearest_euclidean(x, centers),
                                  coordinate_distance_cv(dense, centers)['assignment'])
    # Exact equal centers exercise original first-index tie handling.
    ties = np.vstack((centers[0], centers[0], centers[1]))
    np.testing.assert_array_equal(nearest_euclidean(x, ties),
                                  coordinate_distance_cv(dense, ties)['assignment'])
    agg, error = cv_and_profiles(x, centers, labels, width=3)
    distance = np.abs(dense - centers[labels])
    original_agg = aggregate_distance_groups(distance, labels)
    np.testing.assert_allclose(agg['cv'], original_agg['cv'], rtol=1e-12, atol=1e-12, equal_nan=True)
    np.testing.assert_array_equal(agg['cv_valid'], original_agg['valid'])
    original_profile = corpus_profiles(dense, labels)
    np.testing.assert_allclose(agg['profile_values'], original_profile['values'], rtol=1e-12, atol=1e-12)
    np.testing.assert_array_equal(agg['profile_valid'], original_profile['valid'])
    within, between = bounded_sums_of_squares(x, centers, labels, width=3)
    original_within = float(np.square(dense - centers[labels]).sum())
    mean = centroid_mean(dense)
    original_between = float(sum(15 * np.square(centers[g] - mean).sum() for g in range(3)))
    np.testing.assert_allclose([within, between], [original_within, original_between], rtol=1e-12, atol=1e-12)
    # Preserve the original dense-reference test as an explicit dense test.
    candidate, fitted, membership = fit_candidate(x, 3, n_init=20, fit_storage='dense')
    np.testing.assert_array_equal(membership, coordinate_distance_cv(dense, fitted)['assignment'])
    np.testing.assert_allclose(candidate['CH'], calinski_harabasz_score(dense, membership), rtol=1e-10)
    # The complete fitting path must match the original dense implementation.
    original_fit = ContextDiagnostic.fit(counts, group_counts=[3], random_state=29, n_init=20)
    np.testing.assert_allclose(candidate['CH'], original_fit.variance_ratio, rtol=1e-10, atol=1e-10)
    np.testing.assert_array_equal(membership, original_fit.cluster_labels)
    # CSR storage must satisfy the same geometry and statistics, but it need
    # not reproduce a dense path's local optimum from the same random seed.
    csr_candidate, csr_centers, csr_labels = fit_candidate(x, 3, n_init=20, fit_storage='csr')
    csr_reference = coordinate_distance_cv(dense, csr_centers)
    np.testing.assert_array_equal(csr_labels, csr_reference['assignment'])
    csr_direct_centers = np.stack([centroid_mean(dense[csr_labels == g]) for g in range(3)])
    np.testing.assert_allclose(csr_centers, csr_direct_centers, rtol=1e-13, atol=1e-14)
    np.testing.assert_allclose(csr_candidate['CH'], calinski_harabasz_score(dense, csr_labels), rtol=1e-10, atol=1e-10)
    csr_agg, csr_error = cv_and_profiles(x, csr_centers, csr_labels, width=3)
    csr_distance = np.abs(dense - csr_centers[csr_labels])
    csr_valid = csr_distance.mean(axis=0) > 0
    csr_direct_cv = csr_distance[:, csr_valid].std(axis=0, ddof=0) / csr_distance[:, csr_valid].mean(axis=0)
    np.testing.assert_allclose(csr_agg['cv'][csr_valid], csr_direct_cv, rtol=1e-10, atol=1e-10)
    np.testing.assert_array_equal(csr_agg['cv_valid'], csr_valid)
    # A fixed set of initial centers separates initialization sensitivity from
    # subsequent floating-point Lloyd updates. The observed outcome is recorded
    # without imposing equality as a general property of either storage path.
    initial_centers = dense[[4, 17, 31]].copy()
    fixed_dense, _, fixed_dense_labels = fit_candidate(x, 3, n_init=1, fit_storage='dense', initial_centers=initial_centers)
    fixed_csr, _, fixed_csr_labels = fit_candidate(x, 3, n_init=1, fit_storage='csr', initial_centers=initial_centers)
    sensitivity = dict(
        historical_issue='A previous assertion that same-seed CSR and dense multistart fits must agree failed on this reference. That equality is not a property of the method and is not asserted here.',
        same_seed_multistart=dict(seed=29, n_init=20, dense_CH=candidate['CH'], csr_CH=csr_candidate['CH'],
                                 dense_within_ss=candidate['within_ss'], csr_within_ss=csr_candidate['within_ss'],
                                 identical_assignments=bool(np.array_equal(membership, csr_labels))),
        fixed_initialization=dict(initial_rows=[4, 17, 31], n_init=1, dense_CH=fixed_dense['CH'],
                                  csr_CH=fixed_csr['CH'], dense_within_ss=fixed_dense['within_ss'],
                                  csr_within_ss=fixed_csr['within_ss'],
                                  identical_assignments=bool(np.array_equal(fixed_dense_labels, fixed_csr_labels))),
        consequence='Report the actual CSR storage path and resulting fit. Neither matching nor differing local optima alone establishes algorithmic superiority.')
    available = np.flatnonzero(x.getnnz(axis=1))[:8]
    word_ids = np.array([int(x[i].indices[x[i].data.argmax()]) for i in available])
    original_features = occurrence_features(original_profile, dense[available], word_ids)
    nearest = nearest_euclidean(x[available], agg['profile_means'])
    supplied = np.concatenate((dense[available, word_ids, None],
                               agg['profile_values'][word_ids].reshape(len(available), -1),
                               agg['profile_valid'][word_ids].reshape(len(available), -1).astype(float),
                               agg['profile_between'][word_ids, None],
                               np.abs(dense[available, word_ids] - agg['profile_means'][nearest, word_ids])[:, None]), axis=1)
    np.testing.assert_allclose(supplied, original_features, rtol=1e-12, atol=1e-12)
    # Degenerate measurements retain an unavailable CV, not an invented zero.
    zero_agg = aggregate_distance_groups(np.zeros((6, 2)), np.repeat([0, 1], 3))
    np.testing.assert_array_equal(zero_agg['valid'], [False, False])
    assert np.isnan(zero_agg['cv']).all()
    # Target records remain one-to-one, including empty/OOV processing rows.
    toy = dict(documents=[dict(id=0, text='alpha beta', body_start=0, split='train', label=0),
                          dict(id=1, text='', body_start=0, split='test', label=1)],
               snippets=[dict(doc=0, text='alpha beta', start=0, end=10, split='train', label=0),
                         dict(doc=1, text='', start=0, end=0, split='test', label=1)],
               topics=['a', 'b'])
    validate_corpus(toy)
    target, ids = target_records(toy, sparse.csr_matrix([[.6, .8], [0., 0.]]), np.array(['alpha', 'beta']))
    assert len(target) == 2 and target[0]['word'] == 'beta'
    assert target[0]['start'] == 6 and target[0]['end'] == 10
    assert target[1]['has_target'] is False and ids[1] == -1
    # Compare leading SVD subspaces to the original dense solver. A repeated
    # singular-value pair is fully retained, so individual basis rotations are
    # immaterial while the retained projector is uniquely comparable.
    svd_checks = []
    left, _ = np.linalg.qr(rng.standard_normal((35, 6)))
    right, _ = np.linalg.qr(rng.standard_normal((21, 6)))
    repeated = (left * np.array([9., 7., 7., 5., 3., 1.])) @ right.T
    for matrix, rank in [(dense, 3), (repeated, 3), (np.zeros((8, 5)), 3)]:
        _, s, vt, check = leading_svd(sparse.csr_matrix(matrix), rank)
        _, dense_s, dense_vt = np.linalg.svd(matrix, full_matrices=False)
        np.testing.assert_allclose(s, dense_s[:rank], rtol=1e-10, atol=1e-12)
        if np.any(matrix):
            np.testing.assert_allclose(vt.T @ vt, dense_vt[:rank].T @ dense_vt[:rank],
                                       rtol=1e-9, atol=1e-9)
        svd_checks.append(check)
    return dict(status='PASS', checks=[
        'Exact original IDF and row normalization including zero-IDF axis',
        'Original centroid arithmetic in column batches',
        'Full Euclidean assignments and deterministic exact ties',
        'Original grouped CV including all zero observations and unavailable axes',
        'Independent direct population-distance SD/mean agreement',
        'All eight original profile statistics and validity masks',
        'Batched within/between SS against direct dense arithmetic',
        'Complete dense fit against original dense implementation on declared reference',
        'CSR fitted assignments against original full-vector stable L2 on every reference row',
        'CSR fitted centers against original means on every reference cluster/word',
        'CSR CH against independent sklearn CH and CV against direct pooled distances',
        'Fixed-initialization and same-seed dense/CSR sensitivity recorded without assuming identical optima',
        'CH against sklearn independent implementation',
        'Occurrence features against original occurrence_features',
        'Zero-distance columns retain unavailable CV and a validity mask',
        'Source reconstruction and one-to-one target records including empty text',
        'Leading SVD singular values/projectors versus dense SVD, including repeated spectrum and zero matrix',
    ], cv_max_error=error, csr_cv_max_error=csr_error, dense_reference_selected_k=3,
        svd_checks=svd_checks, storage_sensitivity=sensitivity,
        qualification='Dense-reference equality remains a separately performed test. CSR checks establish the same operations on the declared inputs; they do not claim bit-identical random trajectories or global optimality.')


## Numerical reference construction

Combine the numerical definitions into a resumable calculation over contextual chunks. Cluster selection uses the maximum finite variance ratio criterion across all 99 candidate counts.


In [ ]:
# Purpose: execute the preserved training-only numerical method on every contextual chunk.
def fit_numerical_reference(corpus):
    out = RESULTS_DIRECTORY
    validate_corpus(corpus)
    snippets = corpus['snippets']
    train = np.array([s['split'] == 'train' for s in snippets])
    assert train.sum() > 100, 'Every candidate 2 through 100 requires more than 100 training chunks.'
    identity = object_digest(dict(segmentation=json.loads((out/'segmentation.json').read_text())['identity'],
        partitions=json.loads((out/'source_partitions.json').read_text())['identity'],
        recipe='raw-count-log-n-over-df-l2-lloyd20-seed29-csr-v1', vocabulary='unpruned', k=list(range(2,101)),
        profile_statistics=STATISTICS, svd_rank=32))
    checkpoint = out/'tfidf.json'
    if checkpoint.exists():
        saved = json.loads(checkpoint.read_text())
        assert saved['identity'] == identity
        for name,digest in saved['sha256'].items():
            assert file_sha256(out/name) == digest
        weights = sparse.load_npz(out/'weights.npz')
        arrays = np.load(out/'tfidf_arrays.npz')
        idf, norms = arrays['idf'], arrays['norms']
        vocabulary = np.asarray(json.loads((out/'vocabulary.json').read_text())['words'], dtype=object)
    else:
        texts = [s['text'] for s in snippets]
        vectorizer = CountVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, dtype=np.float64)
        train_counts = vectorizer.fit_transform([text for text,keep in zip(texts,train) if keep])
        counts = vectorizer.transform(texts).tocsr()
        idf = bounded_regular_idf(train_counts)
        weights, norms = bounded_tfidf(counts, idf)
        vocabulary = vectorizer.get_feature_names_out()
        sparse.save_npz(out/'counts.npz', counts)
        sparse.save_npz(out/'weights.npz', weights)
        save_array_file(out/'tfidf_arrays.npz', idf=idf, norms=norms)
        save_json(out/'vocabulary.json', dict(words=vocabulary.tolist()))
        save_json(checkpoint, dict(identity=identity, documents=len(corpus['documents']), chunks=len(snippets),
            training_chunks=int(train.sum()), vocabulary=len(vocabulary), zero_rows=int((norms==0).sum()),
            convention='Raw counts times ln(number of training chunks / training document frequency); no smoothing, no offset; stable row L2 normalization; zero rows retained.',
            sha256={name:file_sha256(out/name) for name in ('counts.npz','weights.npz','tfidf_arrays.npz','vocabulary.json')}))
    x = weights[train]
    x.eliminate_zeros(); x.sort_indices()
    distinct = len({hashlib.sha256(x.indices[x.indptr[i]:x.indptr[i+1]].tobytes()+
                   x.data[x.indptr[i]:x.indptr[i+1]].tobytes()).digest() for i in range(x.shape[0])})
    candidates = out/'cluster_candidates'
    candidates.mkdir(exist_ok=True)
    results = []
    for k in range(2,101):
        check_time_budget()
        path = candidates/f'k_{k:03d}.npz'
        if path.exists():
            with np.load(path) as saved:
                assert str(saved['identity']) == identity
                result = json.loads(str(saved['result_json']))
        else:
            started = time.monotonic()
            if k >= distinct:
                result = dict(k=k,CH=None,status='zero_within_dispersion' if k==distinct else 'insufficient_distinct_rows',
                    within_ss=0.0 if k==distinct else None, between_ss=None, cluster_sizes=[])
                centers = np.empty((0,x.shape[1])); labels = np.empty(0,dtype=int)
            else:
                result,centers,labels = fit_candidate(x,k,n_init=20,seed=29,fit_storage='csr')
            result['seconds'] = time.monotonic()-started
            save_array_file(path, identity=np.array(identity),result_json=np.array(json.dumps(result)),centers=centers,labels=labels)
        results.append(result)
        save_json(out/'cluster_sweep_progress.json', dict(identity=identity,results=results,complete=len(results)==99))
        print(f"Variance ratio criterion: k={k}, score={result['CH']}",flush=True)
    finite = [r for r in results if r['CH'] is not None and np.isfinite(r['CH'])]
    if finite:
        selected = max(finite,key=lambda r:(r['CH'],-r['k']))
        with np.load(candidates/f"k_{selected['k']:03d}.npz") as saved:
            centers,labels = saved['centers'].copy(),saved['labels'].copy()
    else:
        labels = np.zeros(len(x.indptr)-1,dtype=int)
        centers = bounded_centers(x,labels,1)
        selected = dict(k=1,CH=None,cluster_sizes=[x.shape[0]])
    with (out/'cluster_sweep_scores.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=['k','CH','within_ss','between_ss','status'])
        writer.writeheader(); writer.writerows({key:r.get(key) for key in writer.fieldnames} for r in results)
    figure,ax = plt.subplots(figsize=(12,4.5))
    ax.plot([r['k'] for r in results],[np.nan if r['CH'] is None else r['CH'] for r in results],marker='.',markersize=3)
    ax.set_xticks(range(2,101)); ax.tick_params(axis='x',labelrotation=90,labelsize=5)
    ax.set(xlabel='Number of clusters',ylabel='Variance ratio criterion (Caliński–Harabasz index)')
    if finite:
        ax.scatter([selected['k']],[selected['CH']],color='red',zorder=4)
        ax.set_title(f"Selected k={selected['k']}; variance ratio criterion={selected['CH']:.17g}")
    display_figure(figure,'variance_ratio_criterion.png')
    check_time_budget()
    complete_path=out/'cluster_sweep.json'
    if complete_path.exists():
        report=json.loads(complete_path.read_text())
        assert report['identity']==identity and report['complete']
        assert file_sha256(out/'reference.npz')==report['reference_sha256']
        corpus['targets']=json.loads((out/'targets.json').read_text())['targets']
        return report
    assignments=nearest_euclidean(weights,centers)
    np.testing.assert_array_equal(assignments[train],labels)
    profiles,direct_error=cv_and_profiles(x,centers,labels)
    targets,features,word_ids,mismatches=make_targets(corpus,weights,vocabulary,profiles,centers,assignments)
    views,svd_checks=svd_views(x,weights,profiles['cv'],rank=32,seed=29)
    save_array_file(out/'reference.npz',idf=idf,row_norms=norms,centroids=centers,training_mask=train,
        cluster_ids=assignments,training_cluster_ids=labels,features=features,
        occurrence_snippets=np.arange(len(snippets)),target_word_ids=word_ids,target_available=word_ids>=0,**profiles,**views)
    corpus['targets']=targets
    save_json(out/'targets.json',dict(identity=identity,targets=targets))
    report=dict(complete=True,identity=identity,documents=len(corpus['documents']),snippets=len(snippets),
        training_snippets=int(train.sum()),vocabulary=len(vocabulary),selected_k_by_CH=selected['k'],
        selected_CH=selected['CH'],results=results,cluster_count_selection_available=bool(finite),
        selected_cluster_sizes=selected['cluster_sizes'],CV_valid=int(profiles['cv_valid'].sum()),
        CV_unavailable=int((~profiles['cv_valid']).sum()),grouped_vs_direct_CV_max_error=direct_error,
        target_available=int((word_ids>=0).sum()),target_unavailable=int((word_ids<0).sum()),
        zero_tfidf_rows=int((norms==0).sum()),occurrence_feature_width=features.shape[1],svd=svd_checks,
        reference_sha256=file_sha256(out/'reference.npz'),
        fit=dict(algorithm='lloyd',n_init=20,random_state=29,tol=0.0,max_iter=1000,storage='csr'),
        checks=dict(all_99_candidates_completed=len(results)==99,selection_excludes_classification_accuracy=True,
            training_assignments_fixed=True,source_reconstruction=True,grouped_cv_matches_direct=True,
            svd_orthogonality_and_residuals=True,feature_assignment_mismatches=mismatches))
    save_json(complete_path,report)
    return report


## Training and prediction functions

Define aligned token windows, feature scaling, supervised training, source-level predictions and checkpoint recovery. All comparisons use the same source partitions.


In [ ]:
def metrics(y, probabilities, nclasses):
    y = np.asarray(y, dtype=np.int64)
    p = np.asarray(probabilities, dtype=np.float64)
    assert p.shape == (len(y), nclasses) and len(y)
    assert np.isfinite(p).all() and (p >= 0).all()
    np.testing.assert_allclose(p.sum(1), 1, rtol=1e-6, atol=1e-6)
    # Resolve floating-point softmax row-sum roundoff before probability-based metrics.
    p = p / p.sum(axis=1, keepdims=True)
    prediction = p.argmax(1)
    confidence = p.max(1)
    correct = prediction == y
    ece = 0.
    for j in range(10):
        mask = (confidence >= j/10) & (confidence < (j+1)/10 if j < 9 else confidence <= 1)
        if mask.any():
            ece += mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    truth = np.eye(nclasses)[y]
    return dict(accuracy=float(accuracy_score(y, prediction)),
        macro_f1=float(f1_score(y, prediction, labels=np.arange(nclasses), average='macro', zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y, prediction)),
        log_loss=float(log_loss(y, p, labels=np.arange(nclasses))),
        multiclass_brier=float(np.mean(np.sum((p-truth)**2, axis=1))),
        ece_10_bins=float(ece), confusion_matrix=confusion_matrix(y, prediction, labels=np.arange(nclasses)).tolist())

def target_windows(data, tokenizer, max_tokens):
    """Center on the entire target, slicing token IDs only after fixed snippet CV."""
    assert 4 <= max_tokens <= 512
    payload = max_tokens-2
    records = []
    total = 0
    windowed = 0
    for i,t in enumerate(data['targets']):
        available = bool(t.get('has_target', True))
        if not available:
            records.append(dict(target=i, has_target=False, state_start=total, length=0,
                                input_ids=[], target_start=0, target_end=0))
            continue
        text = data['snippets'][t['snippet']]['text']
        encoded = tokenizer(text, add_special_tokens=False, truncation=False, return_offsets_mapping=True, verbose=False)
        ids, offsets = encoded['input_ids'], encoded['offset_mapping']
        hit = [j for j,(start,end) in enumerate(offsets) if end > start and start < t['end'] and end > t['start']]
        assert hit and hit == list(range(hit[0], hit[-1]+1)), (i, t, hit)
        assert offsets[hit[0]][0] == t['start'] and offsets[hit[-1]][1] == t['end'], (i, t, [offsets[j] for j in hit])
        lo, hi = hit[0], hit[-1]+1
        assert hi-lo <= payload, 'Target alone exceeds encoder capacity'
        begin = max(0, lo-(payload-(hi-lo))//2)
        begin = min(begin, max(0, len(ids)-payload))
        end = min(len(ids), begin+payload)
        assert begin <= lo < hi <= end
        selected = [tokenizer.cls_token_id]+ids[begin:end]+[tokenizer.sep_token_id]
        records.append(dict(target=i, has_target=True, state_start=total, length=len(selected),
            input_ids=selected, target_start=lo-begin+1, target_end=hi-begin+1,
            original_wordpieces=len(ids), snippet_token_window_start=begin,
            snippet_token_window_end=end, target_character_start=t['start'], target_character_end=t['end']))
        total += len(selected)
        windowed += int(len(ids) > payload)
    return records, total, windowed

def token_batch(records, indices):
    length = max(records[i]['length'] for i in indices)
    ids = torch.zeros((len(indices), length), dtype=torch.long)
    attention = torch.zeros_like(ids)
    target = torch.zeros_like(ids)
    for j,i in enumerate(indices):
        row = records[i]
        n = row['length']
        ids[j,:n] = torch.tensor(row['input_ids'], dtype=torch.long)
        attention[j,:n] = 1
        target[j,row['target_start']:row['target_end']] = 1
    assert (target <= attention).all() and (target.sum(1) > 0).all()
    return dict(input_ids=ids, attention_mask=attention), target

class Population:
    def __init__(self, data, nclasses, cache):
        self.data = data; self.nclasses = nclasses
        self.docs = {d['id']:d for d in data['documents']}
        self.docids = np.array([t['doc'] for t in data['targets']])
        self.labels = torch.tensor([t['label'] for t in data['targets']],dtype=torch.long)
        self.available = np.array([r['has_target'] for r in cache.records])
        splits = np.array([t['split'] for t in data['targets']])
        self.indices = {s:np.flatnonzero((splits==s)&self.available) for s in ('train','validation','test')}
        train_docs = [d for d in data['documents'] if d['split']=='train']
        counts = np.bincount([d['label'] for d in train_docs], minlength=nclasses)
        assert (counts > 0).all(), 'Each task class needs training-source support'
        self.prior = counts/counts.sum()
        self.fallback = {}

    def aggregate(self, probabilities, indices, split):
        ids = sorted(d['id'] for d in self.data['documents'] if d['split']==split)
        position = {d:i for i,d in enumerate(ids)}
        sums = np.zeros((len(ids),self.nclasses)); counts = np.zeros(len(ids),dtype=np.int64)
        for p,t in zip(probabilities, indices):
            j = position[int(self.docids[t])]; sums[j] += p; counts[j] += 1
        out = np.tile(self.prior,(len(ids),1))
        present = counts > 0
        out[present] = sums[present]/counts[present,None]
        self.fallback[split] = [ids[j] for j in np.flatnonzero(~present)]
        y = np.array([self.docs[d]['label'] for d in ids])
        return ids,y,out

def variant_features(a, name, nrows):
    features = a['features']
    raw = a['raw_svd'][a['occurrence_snippets']]
    cv = a['cv_svd'][a['occurrence_snippets']]
    if name == 'bert_only': return np.empty((nrows,0))
    if name == 'bert_tfidf': return features[:,:1]
    if name == 'bert_profile': return features[:,1:]
    if name == 'bert_full': return features
    if name == 'bert_full_svd': return np.concatenate((features,raw,cv),1)
    if name == 'legacy_frozen_svd_cv': return np.concatenate((raw,cv),1)
    if name == 'legacy_frozen_bert_only': return np.zeros((nrows,raw.shape[1]+cv.shape[1]))
    raise ValueError(name)
# Purpose: use pretrained BERT directly; train its final block and the feature-aware head.
class FrozenPassageBert(nn.Module):
    """Preserved frozen-encoder passage controls, with the same 128-unit prediction head."""
    def __init__(self, bert, width, nclasses, clusters, name):
        super().__init__()
        self.bert=bert.requires_grad_(False)
        self.cluster=nn.Embedding(clusters,16)
        if name=='legacy_frozen_bert_only':
            nn.init.zeros_(self.cluster.weight); self.cluster.requires_grad_(False)
        self.head=nn.Sequential(nn.Linear(bert.config.hidden_size+width+16,128),nn.Tanh(),nn.Linear(128,nclasses))
        self.train(True)
    def train(self, mode=True):
        super().train(mode); self.bert.eval(); return self
    def forward(self,input_ids,attention_mask,target_mask,features,cluster_ids,labels=None):
        with torch.no_grad():
            hidden=self.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state[:,0]
        scores=self.head(torch.cat((hidden,features,self.cluster(cluster_ids)),dim=1))
        return dict(logits=scores,loss=None if labels is None else nn.functional.cross_entropy(scores,labels))

def model_trainable_state(model):
    prefixes=('head.','cluster.') if isinstance(model,FrozenPassageBert) else ('head.',f'bert.encoder.layer.{len(model.bert.encoder.layer)-1}.')
    return {key:value.detach().cpu().clone() for key,value in model.state_dict().items() if key.startswith(prefixes)}

def restore_trainable_state(model,state):
    expected=model_trainable_state(model)
    assert state.keys()==expected.keys()
    model.load_state_dict(state,strict=False)

def prepare_token_windows(corpus,model_folder):
    path=RESULTS_DIRECTORY/'token_windows.json'
    identity=object_digest(dict(targets_sha256=file_sha256(RESULTS_DIRECTORY/'targets.json'),
        revision=BERT_REVISION,max_tokens=512,recipe='whole-target-centered-v1'))
    if path.exists():
        saved=json.loads(path.read_text()); assert saved['identity']==identity
        return saved
    tokenizer=BertTokenizerFast.from_pretrained(model_folder,local_files_only=True)
    records,total,windowed=target_windows(corpus,tokenizer,512)
    saved=dict(identity=identity,records=records,total_tokens=total,windowed_chunks=windowed,
        max_tokens=512,model_revision=BERT_REVISION,
        boundary='Whole selected occurrence centered in at most510 content WordPieces plus CLS and SEP; original computational chunks stay unchanged.')
    save_json(path,saved)
    return saved

def predict_sources(model,features,records,population,cluster_ids,split,batch_size=8):
    model.eval(); probabilities=[]
    indices=population.indices[split]
    with torch.inference_mode():
        for offset in range(0,len(indices),batch_size):
            ii=indices[offset:offset+batch_size].tolist()
            batch,target=token_batch(records,ii)
            arguments={key:value.to(DEVICE) for key,value in batch.items()}
            arguments.update(target_mask=target.to(DEVICE),features=features[ii].to(DEVICE))
            if isinstance(model,FrozenPassageBert):
                arguments['cluster_ids']=torch.as_tensor(cluster_ids[ii],dtype=torch.long,device=DEVICE)
            logits=model(**arguments)['logits']
            probabilities.append(logits.softmax(1).cpu().numpy())
    probabilities=np.concatenate(probabilities) if probabilities else np.empty((0,population.nclasses))
    return population.aggregate(probabilities,indices,split)

def train_comparison(corpus,token_data,model_folder,name,seeds=CLASSIFIER_SEEDS,epochs=2,batch_size=8):
    out=RESULTS_DIRECTORY
    arrays=np.load(out/'reference.npz')
    records=token_data['records']; nclasses=len(corpus['topics'])
    population=Population(corpus,nclasses,SimpleNamespace(records=records))
    training=population.indices['train']
    assert len(training) and len(population.indices['validation']) and len(population.indices['test'])
    features=variant_features(arrays,name,len(records))
    scaler=StandardScaler().fit(features[training]) if features.shape[1] else None
    scaled=torch.as_tensor(scaler.transform(features) if scaler else features,dtype=torch.float32)
    assert torch.isfinite(scaled).all()
    cluster_ids=arrays['cluster_ids'][arrays['occurrence_snippets']]
    configuration=dict(variant=name,seeds=list(seeds),epochs=epochs,batch_size=batch_size,nclasses=nclasses,
        optimizer='AdamW',head_lr=.001,last_block_lr=.00002,weight_decay=.01,hidden_head_width=128,
        max_tokens=512,initial_model_revision=BERT_REVISION,embedding_width=768,
        loss='Ordinary mean cross-entropy over the batch; no source/class weighting',
        selection='Highest validation source macroF1; earliest epoch on ties',
        prediction='Mean eligible target probabilities per source; training-source prior when none',
        feature_scaling='Fit on training occurrences only, after grouped moments and coefficient of variation',
        cache_identity=token_data['identity'],reference_sha256=file_sha256(out/'reference.npz'),
        label_split_sha256=file_sha256(out/'source_partitions.json'),
        encoder_execution='Direct pretrained BertModel forward; bounded batches, no corpus-sized hidden-state cache')
    identity=object_digest(configuration)
    path=out/('part_'+name+'.json')
    if path.exists():
        report=json.loads(path.read_text()); assert report['identity']==identity
    else:
        report=dict(identity=identity,configuration=configuration,results=[],histories=[],predictions={},complete=False)
    completed={r['seed'] for r in report['results']}
    for result in report['results']:
        assert file_sha256(out/result['checkpoint'])==result['checkpoint_sha256']
    if completed==set(seeds):
        return report
    ordered=sorted(training.tolist(),key=lambda i:records[i]['length'])
    batches=[ordered[start:start+batch_size] for start in range(0,len(ordered),batch_size)]
    checkpoint_folder=out/'checkpoints'; checkpoint_folder.mkdir(exist_ok=True)
    for seed in seeds:
        if seed in completed:
            continue
        check_time_budget()
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        bert=BertModel.from_pretrained(model_folder,local_files_only=True,add_pooling_layer=False,attn_implementation='sdpa')
        assert bert.config.hidden_size==768 and bert.config.max_position_embeddings==512 and len(bert.encoder.layer)==12
        frozen_before={key:value.detach().clone() for key,value in bert.embeddings.state_dict().items()}
        initial_last=next(bert.encoder.layer[-1].parameters()).detach().clone()
        legacy=name.startswith('legacy_')
        model=(FrozenPassageBert(bert,features.shape[1],nclasses,len(arrays['centroids']),name) if legacy else
               ContextualBehaviorBert(bert,features.shape[1],nclasses)).to(DEVICE)
        groups=[dict(params=model.head.parameters(),lr=.001)]
        if not legacy:
            groups.append(dict(params=model.bert.encoder.layer[-1].parameters(),lr=.00002))
        elif model.cluster.weight.requires_grad:
            groups.append(dict(params=model.cluster.parameters(),lr=.001))
        optimizer=torch.optim.AdamW(groups,weight_decay=.01)
        resume_path=checkpoint_folder/(name+'__'+str(seed)+'__resume.pt')
        epoch_start=1; batch_start=0; best=-1.0; best_epoch=None; best_state=None
        loss_sum=0.; loss_count=0; max_aux_gradient=0.; histories=[]
        if resume_path.exists():
            saved=torch.load(resume_path,map_location='cpu',weights_only=False)
            assert saved['identity']==identity and saved['seed']==seed
            restore_trainable_state(model,saved['model']); optimizer.load_state_dict(saved['optimizer'])
            epoch_start=saved['epoch']; batch_start=saved['next_batch']
            best=saved['best']; best_epoch=saved['best_epoch']; best_state=saved['best_state']
            loss_sum=saved['loss_sum']; loss_count=saved['loss_count']
            max_aux_gradient=saved['max_aux_gradient']; histories=saved['histories']
            torch.set_rng_state(saved['torch_rng'])
            if DEVICE.type=='cuda': torch.cuda.set_rng_state_all(saved['cuda_rng'])
        generator=np.random.default_rng(seed)
        epoch_orders=[generator.permutation(len(batches)) for _ in range(epochs)]
        started=time.monotonic()
        def save_resume(epoch,next_batch):
            save_torch_file(resume_path,dict(identity=identity,seed=seed,epoch=epoch,next_batch=next_batch,
                model=model_trainable_state(model),optimizer=optimizer.state_dict(),best=best,best_epoch=best_epoch,
                best_state=best_state,loss_sum=loss_sum,loss_count=loss_count,max_aux_gradient=max_aux_gradient,
                histories=histories,torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else []))
        for epoch in range(epoch_start,epochs+1):
            model.train()
            for batch_number in range(batch_start,len(batches)):
                ii=batches[int(epoch_orders[epoch-1][batch_number])]
                batch,target=token_batch(records,ii)
                arguments={key:value.to(DEVICE) for key,value in batch.items()}
                arguments.update(target_mask=target.to(DEVICE),features=scaled[ii].to(DEVICE),labels=population.labels[ii].to(DEVICE))
                if legacy: arguments['cluster_ids']=torch.as_tensor(cluster_ids[ii],dtype=torch.long,device=DEVICE)
                optimizer.zero_grad(set_to_none=True)
                output=model(**arguments); loss=output['loss']; assert torch.isfinite(loss)
                loss.backward()
                if features.shape[1] and name!='legacy_frozen_bert_only':
                    gradient=model.head[0].weight.grad[:,bert.config.hidden_size:bert.config.hidden_size+features.shape[1]]
                    assert torch.isfinite(gradient).all()
                    max_aux_gradient=max(max_aux_gradient,float(gradient.detach().norm()))
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
                optimizer.step()
                loss_sum+=float(loss.detach())*len(ii); loss_count+=len(ii)
                if (batch_number+1)%100==0 or (RUN_BUDGET and time.monotonic()-RUN_STARTED>RUN_BUDGET):
                    save_resume(epoch,batch_number+1)
                    print(f'Training {name}, seed {seed}, epoch {epoch}, batch {batch_number+1}/{len(batches)}',flush=True)
                    check_time_budget()
            assert loss_count==len(training)
            _,validation_labels,probabilities=predict_sources(model,scaled,records,population,cluster_ids,'validation',batch_size)
            score=metrics(validation_labels,probabilities,nclasses)['macro_f1']
            histories.append(dict(variant=name,seed=seed,epoch=epoch,validation_macro_f1=score,training_mean_cross_entropy=loss_sum/loss_count))
            if score>best:
                best=score; best_epoch=epoch; best_state=model_trainable_state(model)
            batch_start=0; loss_sum=0.; loss_count=0
            save_resume(epoch+1,0)
            check_time_budget()
        assert best_state is not None
        restore_trainable_state(model,best_state)
        ids,labels,probabilities=predict_sources(model,scaled,records,population,cluster_ids,'test',batch_size)
        updated=not torch.equal(initial_last,next(model.bert.encoder.layer[-1].parameters()).detach().cpu())
        assert updated if not legacy else not updated
        assert all(torch.equal(value,model.bert.embeddings.state_dict()[key].detach().cpu()) for key,value in frozen_before.items())
        if features.shape[1] and name!='legacy_frozen_bert_only': assert max_aux_gradient>0
        checkpoint=checkpoint_folder/(name+'__'+str(seed)+'.pt')
        save_torch_file(checkpoint,dict(state_dict=best_state,configuration=configuration,seed=seed,
            feature_mean=torch.as_tensor(scaler.mean_) if scaler else torch.empty(0),
            feature_scale=torch.as_tensor(scaler.scale_) if scaler else torch.empty(0)))
        result=dict(variant=name,seed=seed,best_epoch=best_epoch,validation_macro_f1=best,test=metrics(labels,probabilities,nclasses),
            seconds=time.monotonic()-started,last_block_updated=updated,maximum_auxiliary_weight_gradient_norm=max_aux_gradient,
            checkpoint=str(checkpoint.relative_to(out)),checkpoint_sha256=file_sha256(checkpoint))
        report['results'].append(result); report['histories'].extend(histories)
        report['predictions'][name+'__'+str(seed)]=dict(document_ids=ids,labels=labels.tolist(),probabilities=probabilities.tolist())
        report['coverage']=dict(source_documents=len(corpus['documents']),targets=len(records),eligible_targets=int(population.available.sum()),
            unavailable_targets=int((~population.available).sum()),sources_using_prior_by_split=population.fallback,
            training_source_prior=population.prior.tolist())
        report['checks']=dict(all_training_targets_seen_once_each_epoch=True,all_losses_finite=True,all_test_sources_predicted=True,
            embeddings_unchanged=True,earlier_blocks_excluded_from_optimizer=True,feature_weights_received_gradients=True)
        report['complete']={r['seed'] for r in report['results']}==set(seeds)
        save_json(path,report)
        # The finalized checkpoint replaces the temporary optimizer state in history.
        resume_path.unlink(missing_ok=True)
        parts=resume_path.with_name(resume_path.name+'.parts')
        if parts.exists(): shutil.rmtree(parts)
        print(json.dumps(result),flush=True)
        del model,bert,optimizer,best_state,initial_last,frozen_before
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return report

def train_tfidf_baseline(corpus,token_data):
    out=RESULTS_DIRECTORY; path=out/'part_tfidf_logistic.json'
    identity=object_digest(dict(weights=file_sha256(out/'weights.npz'),targets=file_sha256(out/'targets.json'),
        partitions=file_sha256(out/'source_partitions.json'),recipe='logistic-C1-lbfgs1000-seed29-v1'))
    if path.exists():
        report=json.loads(path.read_text()); assert report['identity']==identity and report['complete']; return report
    population=Population(corpus,len(corpus['topics']),SimpleNamespace(records=token_data['records']))
    x=sparse.load_npz(out/'weights.npz')
    tr=population.indices['train']
    classifier=LogisticRegression(C=1.,max_iter=1000,solver='lbfgs',random_state=29)
    classifier.fit(x[tr],population.labels[tr].numpy())
    assert np.array_equal(classifier.classes_,np.arange(population.nclasses))
    assert np.all(classifier.n_iter_<1000),'Logistic optimization reached its declared iteration limit.'
    results={}
    for split in ('validation','test'):
        ii=population.indices[split]
        results[split]=population.aggregate(classifier.predict_proba(x[ii]),ii,split)
    ids,labels,probabilities=results['test']; _,yv,pv=results['validation']
    save_array_file(out/'tfidf_logistic_model.npz',coef=classifier.coef_,intercept=classifier.intercept_,classes=classifier.classes_)
    report=dict(identity=identity,complete=True,
        results=[dict(variant='tfidf_logistic',seed=29,validation_macro_f1=metrics(yv,pv,population.nclasses)['macro_f1'],test=metrics(labels,probabilities,population.nclasses))],
        predictions={'tfidf_logistic':dict(document_ids=ids,labels=labels.tolist(),probabilities=probabilities.tolist())},
        configuration=dict(C=1.,max_iter=1000,solver='lbfgs',random_state=29),
        coverage=dict(sources_using_prior_by_split=population.fallback))
    save_json(path,report); return report


## Comparison and completion checks

Define agreement checks for predictions, paired comparisons and the final verification record. Completion requires the underlying measured artifacts.


In [ ]:
PAIRS=(('bert_full','bert_only'),('bert_full','bert_tfidf'),('bert_profile','bert_only'),('bert_full_svd','bert_full'),('bert_full_svd','bert_only'),('legacy_frozen_svd_cv','legacy_frozen_bert_only'))

def score_predictions(y,prediction,nclasses,weights=None):
    """Rows are true classes; columns are predicted classes; all classes retained."""
    y=np.asarray(y,dtype=np.int64);prediction=np.asarray(prediction,dtype=np.int64)
    assert y.ndim==1 and prediction.shape==y.shape and len(y)
    assert ((0<=y)&(y<nclasses)).all() and ((0<=prediction)&(prediction<nclasses)).all()
    if weights is None: weights=np.ones(len(y),dtype=np.float64)
    weights=np.asarray(weights,dtype=np.float64)
    assert weights.shape==y.shape and np.isfinite(weights).all() and (weights>=0).all() and weights.sum()>0
    matrix=np.bincount(nclasses*y+prediction,weights=weights,minlength=nclasses*nclasses).reshape(nclasses,nclasses)
    true_positive=np.diag(matrix)
    denominator=matrix.sum(1)+matrix.sum(0)
    class_f1=np.divide(2*true_positive,denominator,out=np.zeros(nclasses),where=denominator>0)
    return np.array([true_positive.sum()/matrix.sum(),class_f1.mean()])

def seed_scores(y,predictions,nclasses,weights=None):
    predictions=np.asarray(predictions)
    assert predictions.ndim==2 and predictions.shape[1]==len(y)
    return np.stack([score_predictions(y,p,nclasses,weights) for p in predictions])

def family_structure(families):
    """One family may have many documents; repeated draws repeat every member."""
    unique=[];lookup={};codes=[]
    for family in families:
        key=json.dumps(family,sort_keys=True,ensure_ascii=False)
        if key not in lookup:
            lookup[key]=len(unique);unique.append(family)
        codes.append(lookup[key])
    return unique,np.array(codes,dtype=np.int64)

def family_draw(codes,drawn_families,nfamilies):
    multiplicity=np.bincount(drawn_families,minlength=nfamilies)
    assert len(multiplicity)==nfamilies
    return multiplicity[codes].astype(np.float64)

def paired_bootstrap(y,by_variant,families,nclasses,repetitions=2000,random_seed=907,pairs=PAIRS):
    unique,codes=family_structure(families)
    assert len(unique)>0 and len(codes)==len(y)
    names=sorted({name for pair in pairs for name in pair})
    count=next(iter(by_variant.values())).shape[0]
    assert all(by_variant[name].shape==(count,len(y)) for name in names)
    draws={pair:np.empty((repetitions,2)) for pair in pairs}
    rng=np.random.default_rng(random_seed)
    sample_sizes=np.empty(repetitions,dtype=np.int64)
    absent_class_counts=np.empty(repetitions,dtype=np.int64)
    for b in range(repetitions):
        sampled=rng.integers(0,len(unique),size=len(unique))
        weights=family_draw(codes,sampled,len(unique))
        sample_sizes[b]=int(weights.sum())
        support=np.bincount(y,weights=weights,minlength=nclasses)
        absent_class_counts[b]=int(np.count_nonzero(support==0))
        averages={name:seed_scores(y,by_variant[name],nclasses,weights).mean(0) for name in names}
        for left,right in pairs:
            draws[(left,right)][b]=averages[left]-averages[right]
    comparisons=[]
    for left,right in pairs:
        individual=seed_scores(y,by_variant[left],nclasses)-seed_scores(y,by_variant[right],nclasses)
        observed=individual.mean(0)
        interval=np.quantile(draws[(left,right)],[.025,.975],axis=0,method='linear')
        statistics={}
        for j,metric in enumerate(('accuracy','macro_f1')):
            lo,hi=interval[:,j]
            relation='contains_zero'
            if lo>0: relation='above_zero'
            elif hi<0: relation='below_zero'
            statistics[metric]=dict(mean_seed_difference=float(observed[j]),
                individual_seed_differences=individual[:,j].tolist(),
                individual_seed_difference_sd=float(individual[:,j].std(ddof=1)) if count>1 else None,
                percentile_95_interval=[float(lo),float(hi)],interval_relative_to_zero=relation)
        comparisons.append(dict(left=left,right=right,difference='left minus right',statistics=statistics))
    return dict(comparisons=comparisons,
        bootstrap=dict(repetitions=repetitions,random_seed=random_seed,unit='source family',
            families=len(unique),draws_per_replicate=len(unique),sampling='Uniform families with replacement; retain every member of each drawn family',
            pairing='Same family multiplicities for every compared model and fixed seed',
            statistic='Arithmetic mean of individual-seed scores; no probability or label ensemble',
            interval='Pointwise95%percentile interval; quantiles0.025and0.975 with linear interpolation',
            replicate_document_count_min=int(sample_sizes.min()),replicate_document_count_max=int(sample_sizes.max()),
            replicates_with_absent_classes=int(np.count_nonzero(absent_class_counts)),
            maximum_absent_classes=int(absent_class_counts.max()),
            absent_class_rule='MacroF1 retains the full task class inventory; zero denominator contributes0'))
# Purpose: check every comparison against the same held-out source identities and measured predictions.
PAIRS=(('bert_full','bert_only'),('bert_full','bert_tfidf'),('bert_profile','bert_only'),
       ('bert_full_svd','bert_full'),('bert_full_svd','bert_only'),
       ('legacy_frozen_svd_cv','legacy_frozen_bert_only'))

def summarize_classification(corpus,repetitions=2000):
    out=RESULTS_DIRECTORY
    test=sorted((d for d in corpus['documents'] if d['split']=='test'),key=lambda d:d['id'])
    ids=[d['id'] for d in test]; labels=np.array([d['label'] for d in test])
    nclasses=len(corpus['topics']); summaries={}; predictions={}; audits={}; configurations={}
    for name in (*CLASSIFIERS,'tfidf_logistic'):
        report=json.loads((out/('part_'+name+'.json')).read_text())
        assert report['complete'] is True
        expected=(29,) if name=='tfidf_logistic' else CLASSIFIER_SEEDS
        assert len(report['results'])==len(expected) and {r['seed'] for r in report['results']}==set(expected)
        required={name if name=='tfidf_logistic' else name+'__'+str(seed) for seed in expected}
        assert set(report['predictions'])==required
        row_values=[]; row_predictions=[]
        for seed in expected:
            key=name if name=='tfidf_logistic' else name+'__'+str(seed)
            recorded=report['predictions'][key]
            assert recorded['document_ids']==ids and recorded['labels']==labels.tolist()
            probability=np.asarray(recorded['probabilities'])
            recomputed=metrics(labels,probability,nclasses)
            result=next(r for r in report['results'] if r['seed']==seed)
            for measure in ('accuracy','macro_f1'):
                np.testing.assert_allclose(recomputed[measure],result['test'][measure],rtol=1e-11,atol=1e-12)
            row_values.append([recomputed['accuracy'],recomputed['macro_f1']])
            row_predictions.append(probability.argmax(1))
        values=np.asarray(row_values)
        summaries[name]=dict(seeds=list(expected),runs=len(expected),test_documents=len(ids),
            metrics={metric:dict(mean=float(values[:,i].mean()),sd=float(values[:,i].std(ddof=1)) if len(expected)>1 else None,
                                 individual_seed_values=values[:,i].tolist()) for i,metric in enumerate(('accuracy','macro_f1'))})
        predictions[name]=np.stack(row_predictions)
        configurations[name]=report['configuration']
        audits[name]=dict(all_heldout_document_ids_and_labels_identical=True,stored_metrics_match_recomputed_scores=True,complete=True)
    for field in ('seeds','epochs','batch_size','nclasses','max_tokens','initial_model_revision','reference_sha256','label_split_sha256'):
        assert all(configurations[name][field]==configurations[CLASSIFIERS[0]][field] for name in CLASSIFIERS)
    bootstrap=paired_bootstrap(labels,predictions,[d['family'] for d in test],nclasses,
                               repetitions=repetitions,random_seed=907,pairs=PAIRS)
    report=dict(complete=True,summary=summaries,paired_comparisons=bootstrap['comparisons'],bootstrap=bootstrap['bootstrap'],
        test_documents=len(ids),test_document_ids=ids,test_labels=labels.tolist(),task_classes=nclasses,seeds=list(CLASSIFIER_SEEDS),checks=audits,
        primary_comparison=dict(left='bert_full',right='bert_tfidf'),
        task='Primary arXiv category classification; source labels inherited by target occurrences.',
        selection='No held-out performance-based changes, removals or selective reruns.',
        uncertainty_scope='Paired source-family bootstrap, conditional on this split and these fitted seeds; pointwise intervals.')
    save_json(out/'classification_summary.json',report)
    figure,ax=plt.subplots(figsize=(11,4.5))
    names=list(summaries)
    ax.bar(np.arange(len(names))-.18,[summaries[n]['metrics']['accuracy']['mean'] for n in names],width=.36,label='Accuracy')
    ax.bar(np.arange(len(names))+.18,[summaries[n]['metrics']['macro_f1']['mean'] for n in names],width=.36,label='Macro F1')
    ax.set_xticks(np.arange(len(names)),names,rotation=35,ha='right'); ax.set_ylim(0,1)
    ax.set_ylabel('Held-out source classification'); ax.legend(); figure.tight_layout()
    display_figure(figure,'classification_comparisons.png')
    return report

def verify_complete_execution(corpus):
    out=RESULTS_DIRECTORY
    validate_corpus(corpus)
    assert len(corpus['documents'])==8836
    segmentation=json.loads((out/'segmentation.json').read_text())
    numerical=json.loads((out/'cluster_sweep.json').read_text())
    classification=json.loads((out/'classification_summary.json').read_text())
    assert segmentation['complete'] and numerical['complete'] and classification['complete']
    assert segmentation['snippets']==numerical['snippets']==len(corpus['snippets'])
    assert [r['k'] for r in numerical['results']]==list(range(2,101))
    valid=[r for r in numerical['results'] if r['CH'] is not None and np.isfinite(r['CH'])]
    if valid:
        best=max(valid,key=lambda r:(r['CH'],-r['k']))
        assert numerical['selected_k_by_CH']==best['k'] and numerical['selected_CH']==best['CH']
    assert file_sha256(out/'reference.npz')==numerical['reference_sha256']
    assert all(r['complete'] for r in classification['checks'].values())
    report=dict(complete=True,dataset_revision=DATASET_REVISION,source_documents=len(corpus['documents']),
        chunks=len(corpus['snippets']),selected_cluster_count=numerical['selected_k_by_CH'],
        exact_variance_ratio=numerical['selected_CH'],
        checks=dict(complete_source_coverage=True,contiguous_chunk_coverage=True,training_only_numerical_fit=True,
            all_99_cluster_counts=True,fixed_centroids_and_assignments=True,grouped_moments_checked=True,
            all_declared_classifier_seeds=True,all_test_sources_predicted=True,stored_metrics_recomputed=True),
        classification_comparisons=classification['paired_comparisons'],
        performance_statement='Measured comparisons are separate from implementation validity; signs are preserved exactly.',
        scientific_code_sha256=code_fingerprint(),
        evidence_sha256={name:file_sha256(out/name) for name in ('source_partitions.json','segmentation.json','tfidf.json',
             'cluster_sweep.json','classification_summary.json','numerical_self_test.json')})
    save_json(out/'verification.json',report)
    print(json.dumps(report,indent=2),flush=True)
    return report


## Source partitions and pretrained weights

Restore stored artifacts, establish the source partitions and obtain the pinned segmentation and BERT checkpoints used by subsequent execution.

Keep every source and group exact duplicate texts or equal nonempty digital object identifiers before splitting. Five stratified group folds use seed 29: one test fold, one validation fold and three training folds. If a rare category has no training source, move one deterministic whole family to training and record the adjustment. No document is removed. Download the already pretrained models at their saved revisions.


In [ ]:
if DATASET_AVAILABLE:
    restore_large_files(EXPERIMENT_DIRECTORY)
    partitions = partition_sources()
    print(json.dumps({'documents':len(partitions['documents']),'classes':len(partitions['topics']), 'source_counts':dict(Counter(d['split'] for d in partitions['documents'])), 'training_support_adjustments':partitions['training_support_adjustments']},indent=2))
    MODEL_DIRECTORY=EXPERIMENT_DIRECTORY/'models'
    SEGMENTER_DIRECTORY=obtain_pretrained_model(SEGMENTER_REPOSITORY,SEGMENTER_REVISION,MODEL_DIRECTORY/'context_segmenter')
    BERT_DIRECTORY=obtain_pretrained_model(BERT_REPOSITORY,BERT_REVISION,MODEL_DIRECTORY/'bert')
    pack_large_files(MODEL_DIRECTORY)
else:
    print("Skipped: dataset not found.")


## Contiguous contextual chunks

Apply the pretrained separator model to every normalized source. Verify that chunk boundaries cover each source position exactly once.

Chonky predicts separator spans using the saved pretrained token classifier. Its original splitter places boundaries at separator ends and retains the final tail. Overlapping inference windows have 1,024 positions and stride 512; their edges do not become chunk boundaries. Concatenating each source’s chunks must recover every normalized character exactly once.


In [ ]:
if DATASET_AVAILABLE:
    segmentation_report = segment_sources(partitions, SEGMENTER_DIRECTORY)
    corpus = load_corpus()
    print(json.dumps(segmentation_report,indent=2))
else:
    print("Skipped: dataset not found.")


## Cluster selection and contextual profiles

Check the numerical definitions, evaluate cluster counts 2 through 100, and construct the selected reference, grouped moments, contextual profiles and singular value decompositions.

Training chunks alone define vocabulary, unsmoothed logarithmic inverse document frequencies, Euclidean k-means and profiles. Examine every cluster count from 2 through 100 with 20 initializations and seed 29. Select the largest finite variance ratio criterion, also called the Caliński–Harabasz index or pseudo-F statistic; prefer the smaller count on an exact tie. Classification accuracy never selects the count. Calculate coordinate-distance moments, the per-word coefficient of variation and the full contextual profile after fixing the selected assignments. Apply singular value decomposition afterward.


In [ ]:
if DATASET_AVAILABLE:
    numerical_checks = numerical_self_test()
    save_json(RESULTS_DIRECTORY/'numerical_self_test.json',numerical_checks)
    numerical_report = fit_numerical_reference(corpus)
    print(json.dumps({key:numerical_report[key] for key in ('documents','snippets','vocabulary','selected_k_by_CH','selected_CH')},indent=2))
else:
    print("Skipped: dataset not found.")


## Token alignment and lexical baseline

Prepare the target-centered token windows used by the transformer comparisons and evaluate logistic regression on the training-fitted lexical representation.

Each target word’s character span is aligned with its WordPieces. A whole-target-centered window respects BERT’s 512-position limit without changing the computational chunk. The scalar local term-frequency–inverse-document-frequency weight and full contextual profile join the target’s contextual BERT state at the classification head. The final encoder block and head train jointly using ordinary mean cross-entropy. The same source partitions, target windows, seeds, epochs and optimizer settings apply to all comparisons. Validation selects the epoch; test labels are used only for evaluation.


In [ ]:
if DATASET_AVAILABLE:
    token_data = prepare_token_windows(corpus, BERT_DIRECTORY)
    logistic_report = train_tfidf_baseline(corpus, token_data)
else:
    print("Skipped: dataset not found.")


## BERT baseline

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified bert_only comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'bert_only')
else:
    print("Skipped: dataset not found.")


## BERT with term frequency–inverse document frequency

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified bert_tfidf comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'bert_tfidf')
else:
    print("Skipped: dataset not found.")


## BERT with contextual profiles

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified bert_profile comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'bert_profile')
else:
    print("Skipped: dataset not found.")


## BERT with lexical weights and contextual profiles

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified bert_full comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'bert_full')
else:
    print("Skipped: dataset not found.")


## BERT with singular value decomposition features

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified bert_full_svd comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'bert_full_svd')
else:
    print("Skipped: dataset not found.")


## Frozen BERT baseline

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified legacy_frozen_bert_only comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'legacy_frozen_bert_only')
else:
    print("Skipped: dataset not found.")


## Frozen BERT with weighted decomposition and cluster identity

Train and evaluate this prescribed comparison using the aligned inputs and fixed source partitions. Save each seed’s predictions and resumable state for the final comparison.


In [ ]:
if DATASET_AVAILABLE:
    # Purpose: execute the prespecified legacy_frozen_svd_cv comparison with all three seeds.
    train_comparison(corpus, token_data, BERT_DIRECTORY, 'legacy_frozen_svd_cv')
else:
    print("Skipped: dataset not found.")


## Measured comparisons and final verification

Summarize the completed classifiers and validate their shared source population, numerical artifacts and coverage. Write the completion record only when those checks pass.

Recompute reported metrics from complete held-out source predictions. Compare the same sources and seeds, including the contextual aggregation comparison against BERT with only the local term-frequency–inverse-document-frequency feature. Preserve the measured direction of every comparison. Successful implementation and better classification performance are assessed separately.


In [ ]:
if DATASET_AVAILABLE:
    classification_summary = summarize_classification(corpus)
    verification = verify_complete_execution(corpus)
    pack_large_files(EXPERIMENT_DIRECTORY)
else:
    print("Skipped: dataset not found.")
